# WASAC Smart Water Billing: Meter Reading Pipeline

**Author:** Dushime Paulette, BSc Software Engineering (Machine Learning), African Leadership University
**Supervisor:** Samiratu Ntoshi

## What this notebook does

A household takes a photo of its water meter. This notebook builds and tests the machine learning models that turn that photo into a reading the billing system can trust.

| Stage | Job | Model in this notebook |
|---|---|---|
| 0. Find the display | Cut the number window out of the full photo | YOLOv8 (Benax, already trained, used as-is) |
| 1. Quality gate | Warn when a photo is too poor to read | MobileNetV2 (trained here) |
| 2. Meter reading | Read the digits in the number window | CRNN + CTC (trained here), compared with ResNet18 + BiGRU |
| 3. Anomaly check | Compare the reading with the household's history | Isolation Forest (lives in the backend, see the last section) |

## How the notebook is organised

It follows the standard machine learning workflow, in this order:

1. **Setup**: libraries, paths, settings.
2. **Data collection**: load the three data sources.
3. **Data preparation**: cut out the meter displays and clean the labels.
4. **Exploratory data analysis (EDA)**: understand the data before modelling.
5. **Data splitting**: train, validation and test sets, split *by meter* so nothing leaks.
6. **Preprocessing and feature engineering**: turning images into model inputs, plus augmentation.
7. **Baselines**: simple methods any new model must beat.
8. **Model architecture**: the CRNN and the alternative model.
9. **Training**: pretraining on Yandex, then fine-tuning on Kigali meters.
10. **Error analysis**: what kinds of mistakes the model still makes.
11. **Alternative model and model selection**: chosen on validation only.
12. **Final evaluation**: the locked test set, used once.
13. **Stage 1 quality gate**: trained and evaluated the same way.
14. **Export for deployment**: files the FastAPI backend needs.
15. **Summary, limitations and next steps.**

## Before you run it

1. **Settings (right panel):** Accelerator = **GPU T4 x2** or **P100**; Internet = **On** (needed to install
   EasyOCR and download ImageNet weights).
2. **Inputs attached:** the Benax YOLOv8 model, the raw Kigali photos, the verified labels dataset, and the
   Yandex Toloka water meters dataset. The path check in Section 1 tells you if anything is missing.
3. **Run it with "Save Version → Save & Run All (Commit)"**, not cell by cell. A committed run keeps every
   output file permanently, and every number in the report then comes from one run that anyone can repeat.
   It takes roughly 50–70 minutes on a GPU.
4. **Never edit and re-run single cells after the final test section** and then report those numbers.
   The test set is only fair if it is used once.

# 1. Setup

## 1.1 Install missing libraries

Kaggle already has PyTorch, OpenCV, pandas and scikit-learn. Two libraries are sometimes missing:

- **ultralytics** runs the YOLOv8 model that finds the number window in each photo.
- **easyocr** is a ready-made text reader. We use it only as a baseline to show why a custom model is needed.

The cell only installs a library if it is not already there, so it is safe to run again.

In [ ]:
import importlib.util, subprocess, sys

for package, module in [("ultralytics", "ultralytics"), ("easyocr", "easyocr")]:
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
    else:
        print(f"{package} already installed")

## 1.2 Imports, paths and settings

Everything you might need to change lives in this one cell:

- **Paths** to the Kaggle inputs and to the output folders.
- **`SEED = 42`** fixes the random choices (data splits, shuffling, weight initialisation) so the
  notebook gives the same splits every time. GPU training still varies very slightly between runs,
  which is normal; this is why the report uses the numbers from one committed run.
- **`EPOCHS`**: how many passes over the training data each model gets.
- **`FAST_RUN`**: a test switch that trains every model for only 2 epochs, used to check that the code
  runs end to end. It is **off** on Kaggle. Never report numbers from a fast run.

The last part checks that every input file exists and stops with a clear message if one is missing.

In [ ]:
import os, re, ast, json, math, copy, time, random, shutil, warnings
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = str(2 ** 40)   # some phone photos are very large

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from scipy.stats import rankdata

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---------------- Inputs (Kaggle datasets and models) ----------------
YOLO_MODEL_PATH    = "/kaggle/input/models/paulettedushime/best-model/tensorflow2/default/1/best.pt"
KIGALI_IMAGES_DIR  = "/kaggle/input/datasets/paulettedushime/images-for-water-meters/images"
KIGALI_LABELS_PATH = "/kaggle/input/datasets/paulettedushime/rwanda-water-meters-verified/rwanda_crnn_training_labels.csv"
UNREADABLE_PATH    = "/kaggle/input/datasets/paulettedushime/rwanda-water-meters-verified/rwanda_unreadable_labels.csv"
YANDEX_DIR         = "/kaggle/input/datasets/tapakah68/yandextoloka-water-meters-dataset/WaterMeters"
YANDEX_IMAGES_DIR  = os.path.join(YANDEX_DIR, "images")
YANDEX_CSV         = os.path.join(YANDEX_DIR, "data.csv")

# ---------------- Outputs (everything saved under /kaggle/working) ----------------
WORK         = "/kaggle/working"
CROPS_DIR    = os.path.join(WORK, "kigali_crops")          # Kigali number windows
YANDEX_CROPS = os.path.join(WORK, "yandex_crops")          # Yandex number windows (before orientation fix)
ORIENTED_DIR = os.path.join(WORK, "yandex_crops_oriented") # Yandex number windows (after orientation fix)
MODELS_DIR   = os.path.join(WORK, "models")
PLOTS_DIR    = os.path.join(WORK, "plots")
RESULTS_DIR  = os.path.join(WORK, "results")
SPLITS_DIR   = os.path.join(WORK, "splits")
DEPLOY_DIR   = os.path.join(WORK, "deploy")
for d in [CROPS_DIR, YANDEX_CROPS, ORIENTED_DIR, MODELS_DIR, PLOTS_DIR, RESULTS_DIR, SPLITS_DIR, DEPLOY_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------------- Settings ----------------
SEED = 42
FAST_RUN = os.environ.get("WASAC_FAST_RUN") == "1"   # test switch, off on Kaggle
EPOCHS = dict(yandex_pretrain=45, kigali_scratch=60, kigali_finetune=60, resnet_pretrain=25, quality_gate=30)
if FAST_RUN:
    EPOCHS = {k: 2 for k in EPOCHS}
N_BOOTSTRAP = 200 if FAST_RUN else 2000
NUM_WORKERS = 2

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

# ---------------- Check inputs ----------------
print("=" * 70)
print("INPUT CHECK")
print("=" * 70)
missing = []
for name, path in [("YOLOv8 model", YOLO_MODEL_PATH), ("Kigali photos", KIGALI_IMAGES_DIR),
                   ("Kigali labels", KIGALI_LABELS_PATH), ("Unreadable list", UNREADABLE_PATH),
                   ("Yandex images", YANDEX_IMAGES_DIR), ("Yandex CSV", YANDEX_CSV)]:
    ok = os.path.exists(path)
    print(f"  {'OK     ' if ok else 'MISSING'}  {name:16s} {path}")
    if not ok:
        missing.append(name)
if missing:
    raise FileNotFoundError(f"Attach these inputs before running: {missing}")

print(f"\nDevice: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
print(f"FAST_RUN = {FAST_RUN}  (must be False for reported results)")
if DEVICE.type != "cuda" and not FAST_RUN:
    print("WARNING: no GPU found. Turn on the GPU accelerator, otherwise training will take hours.")

## 1.3 Helper functions for measuring accuracy

Before any modelling, we define **how a reading is judged**. These definitions are used in every
section, so all numbers in the notebook are comparable.

| Metric | Meaning | Why we use it |
|---|---|---|
| **Exact match** | Every digit, including leading zeros, is identical | Strictest test |
| **Reading accuracy** (headline) | Identical after removing leading zeros (`00565846` = `565846`) | Leading zeros never change a bill, and the labels' zeros are not fully reliable (Section 9.3) |
| **Per-meter accuracy** | Reading accuracy averaged per meter, so each meter counts once | Some meters were photographed up to 14 times; this stops them from dominating |
| **Edit distance** | How many digits must be added, removed or changed to fix the reading | Shows how *close* a wrong reading is |
| **Correct or within one digit** | Right, or wrong in exactly one position | Most single-digit misses are the last litre wheel caught mid-turn, which barely affects a bill |
| **Length errors** | The model dropped or added a digit | A specific weakness we track |
| **95% confidence interval** | The range the true accuracy probably lies in | Our test set is small, so we must show the uncertainty |

The confidence interval uses a **meter-level bootstrap**: we resample whole meters (not single photos)
2,000 times and see how much the accuracy moves. Resampling meters is the honest choice, because
photos of the same meter are not independent.

In [ ]:
def digits_only(s):
    # Keep only the digits of a label, e.g. ' 00565846 ' -> '00565846'.
    return re.sub(r"\D", "", str(s))

def strip0(s):
    # Remove leading zeros, keeping a single '0' for an all-zero reading.
    s = str(s).lstrip("0")
    return s if s else "0"

def edit_distance(a, b):
    # Levenshtein distance: insertions, deletions and substitutions needed to turn a into b.
    a, b = str(a), str(b)
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]

def within_one_digit(pred, true):
    p, t = strip0(pred), strip0(true)
    return p == t or (len(p) == len(t) and sum(x != y for x, y in zip(p, t)) == 1)

def reading_metrics(preds, gts, groups=None):
    # All accuracy numbers used in the notebook, as percentages.
    ok = np.array([strip0(p) == strip0(g) for p, g in zip(preds, gts)])
    out = {
        "exact %":   float(100 * np.mean([p == g for p, g in zip(preds, gts)])),
        "reading %": float(100 * ok.mean()),
        "edit dist": float(np.mean([edit_distance(strip0(p), strip0(g)) for p, g in zip(preds, gts)])),
        "within-1 %": float(100 * np.mean([within_one_digit(p, g) for p, g in zip(preds, gts)])),
        "length errs": int(sum(len(strip0(p)) != len(strip0(g)) for p, g in zip(preds, gts))),
    }
    if groups is not None:
        out["per-meter %"] = float(100 * pd.Series(ok).groupby(np.asarray(groups)).mean().mean())
    return out

def bootstrap_ci(ok, groups, n=None, seed=SEED):
    # 95% confidence interval for accuracy, resampling whole meters.
    n = n or N_BOOTSTRAP
    rng = np.random.default_rng(seed)
    g = pd.DataFrame({"ok": np.asarray(ok, float), "g": np.asarray(groups)}).groupby("g")["ok"]
    sums, counts = g.sum().values, g.count().values
    boots = [sums[idx].sum() / counts[idx].sum()
             for idx in (rng.integers(0, len(sums), len(sums)) for _ in range(n))]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return 100 * lo, 100 * hi

def save_plot(name):
    path = os.path.join(PLOTS_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved plot: {path}")

def fmt(d):
    return " | ".join(f"{k} {v:.1f}" if isinstance(v, float) else f"{k} {v}" for k, v in d.items())

print("Helpers ready. Example:", reading_metrics(["00565846", "0564846"], ["565846", "00565846"]))

---
# 2. Data collection

The project uses three real data sources. No synthetic data is used for training the reader.

| Source | What it is | Size | Used for |
|---|---|---|---|
| **Kigali (Benax) photos** | Real photos of WASAC meters in Kigali, shared by the developer of a public Rwandan YOLOv8 meter model for academic use | 652 photos | Fine-tuning and testing on *our* meters |
| **Verified labels** | I checked every crop by eye: confirmed the reading, corrected it, or marked it unreadable | 536 readings + 98 unreadable | Ground truth for Kigali |
| **Yandex Toloka water meters** | A public dataset of 1,244 meter photos from Russia (CC BY-NC-ND 4.0) | 1,244 photos | Pretraining: teaching the model what digits on meter wheels look like |

**Why two datasets?** 536 Kigali readings are too few to train a reading model from nothing. The usual
solution is **transfer learning**: learn general skills (finding and reading digit wheels) on a bigger
dataset first, then adapt to the target meters. Yandex meters look different from Kigali meters (other
brands, other photos), and measuring that difference, the **domain gap**, is one of the findings of this
project.

### Label files

- `rwanda_crnn_training_labels.csv`: one row per readable crop with `crop_filename`, `original_filename`,
  `label` (the verified reading) and `verification_status`:
  - `VERIFIED_CORRECT`: the YOLO model's reading was right and I confirmed it.
  - `VERIFIED_CORRECTED`: the YOLO model was wrong and I typed the correct reading.
  - `VERIFIED_PARTIAL`: part of the display (usually the last wheel) was cut off.
- `rwanda_unreadable_labels.csv`: 98 crops I could not read, with the reason (dirt, condensation,
  wheels mid-turn, bad angle). These train the quality gate.

The labels are loaded **as text, not numbers**. Loading `00565846` as a number would silently turn it
into `565846`. This exact problem damaged some labels before they reached this notebook (Section 4.2).

In [ ]:
kigali_df = pd.read_csv(KIGALI_LABELS_PATH, dtype=str).fillna("")
required = {"crop_filename", "original_filename", "label", "verification_status"}
assert required <= set(kigali_df.columns), f"Label CSV must contain {required}"
kigali_df["label_raw"] = kigali_df["label"].map(digits_only)
n_before = len(kigali_df)
kigali_df = kigali_df[kigali_df["label_raw"].str.len() > 0].reset_index(drop=True)

unread_df = pd.read_csv(UNREADABLE_PATH, dtype=str).fillna("")
assert "crop_filename" in unread_df.columns
if "reason" not in unread_df.columns:
    unread_df["reason"] = "unreadable"

yandex_raw = pd.read_csv(YANDEX_CSV)
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
raw_photos = sorted(f for f in os.listdir(KIGALI_IMAGES_DIR) if os.path.splitext(f)[1].lower() in VALID_EXT)

overlap = set(kigali_df["crop_filename"]) & set(unread_df["crop_filename"])
print("=" * 70)
print("DATA COLLECTED")
print("=" * 70)
print(f"Kigali raw photos:          {len(raw_photos)}")
print(f"Kigali readable labels:     {len(kigali_df)}  (dropped {n_before - len(kigali_df)} empty labels)")
print(f"Kigali unreadable crops:    {len(unread_df)}")
print(f"Crops in both lists:        {len(overlap)}  (should be 0)")
print(f"Yandex photos:              {len(yandex_raw)}  columns: {list(yandex_raw.columns)}")
print("\nVerification status:")
print(kigali_df["verification_status"].value_counts().to_string())
kigali_df.head()

---
# 3. Data preparation

The reading model does not look at the whole photo. It looks only at the **number window**, the strip of
rolling digit wheels. So the first job is to cut that strip out of every photo. We also clean the Yandex
labels, which turned out to be the cause of an early training failure.

## 3.1 Shared preprocessing code (`wasac_preprocessing.py`)

The code that cuts out the number window must be **identical** in training and in the real app; if the
backend crops photos differently, the model sees different inputs and its accuracy drops. So this cell
writes the code into a small Python file. The notebook imports it, and the same file is copied to the
backend later (Section 14).

**How the crop works:** the Benax YOLOv8 model detects two kinds of objects: the **display window**
(class 1) and **individual digits** (classes 2–11 for 0–9).

1. If a display window is found, take the most confident one.
2. Otherwise, draw one box around all detected digits.
3. Otherwise, keep the full photo (rare).
4. Add 5% padding on each side so edge digits are never cut.

This is the same procedure that produced the crops I verified by hand, so the labels match the crops.

In [ ]:
%%writefile /kaggle/working/deploy/wasac_preprocessing.py
"""Shared preprocessing for the WASAC meter pipeline (training notebook and FastAPI backend)."""
import numpy as np

DISPLAY_CLASS = 1                       # YOLO class id of the meter display window
DIGIT_CLASSES = set(range(2, 12))       # YOLO class ids 2..11 are the digits 0..9
READING_LENGTH = 8                      # 5 cubic-metre wheels + 3 litre wheels
DECIMALS = 3


def crop_display(img_bgr, yolo_model, conf=0.15, imgsz=640, device=None, pad=0.05):
    """Cut the number window out of a full meter photo.

    Returns (crop_bgr, method) where method is 'window', 'digits', 'full' or 'none'.
    """
    h, w = img_bgr.shape[:2]
    result = yolo_model.predict(source=img_bgr, imgsz=imgsz, conf=conf, device=device, verbose=False)[0]
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return None, "none"
    cls = boxes.cls.cpu().numpy().astype(int)
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    windows = [i for i, c in enumerate(cls) if c == DISPLAY_CLASS]
    if windows:
        best = max(windows, key=lambda i: confs[i])
        x1, y1, x2, y2 = map(int, xyxy[best])
        method = "window"
    else:
        digits = [xyxy[i] for i, c in enumerate(cls) if c in DIGIT_CLASSES]
        if digits:
            x1 = int(min(b[0] for b in digits)); y1 = int(min(b[1] for b in digits))
            x2 = int(max(b[2] for b in digits)); y2 = int(max(b[3] for b in digits))
            method = "digits"
        else:
            x1, y1, x2, y2 = 0, 0, w, h
            method = "full"

    px, py = int((x2 - x1) * pad), int((y2 - y1) * pad)
    x1, y1 = max(0, x1 - px), max(0, y1 - py)
    x2, y2 = min(w, x2 + px), min(h, y2 + py)
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None, "none"
    return crop, method


def pad_reading(digits, length=READING_LENGTH):
    """Add the leading zeros the reader sometimes drops: '565846' -> '00565846'.

    Returns None when the reading has more digits than the meter has wheels,
    so the app can ask the household to check the numbers instead of guessing.
    """
    digits = "".join(ch for ch in str(digits) if ch.isdigit())
    if not digits or len(digits) > length:
        return None
    return digits.zfill(length)


def digits_to_m3(padded_digits, decimals=DECIMALS):
    """'00565846' -> 565.846 cubic metres."""
    if padded_digits is None:
        return None
    return int(padded_digits[:-decimals]) + int(padded_digits[-decimals:]) / 10 ** decimals

In [ ]:
import importlib
if DEPLOY_DIR not in sys.path:
    sys.path.insert(0, DEPLOY_DIR)
import wasac_preprocessing as wp
importlib.reload(wp)
print("pad_reading('565846') ->", wp.pad_reading("565846"), "| m3:", wp.digits_to_m3(wp.pad_reading("565846")))

## 3.2 Cut out the number window from every Kigali photo

This runs YOLOv8 on all 652 photos and saves each crop as `<photo name>_crop.jpg`, the same names used
in the label files. If a crop already exists it is skipped, so re-running is fast.

After cropping we check that **every labelled crop and every unreadable crop now exists**. A label with no
image cannot be used, so it is reported and removed.

In [ ]:
from ultralytics import YOLO

yolo = YOLO(YOLO_MODEL_PATH)
yolo_device = 0 if DEVICE.type == "cuda" else "cpu"

records, t0 = [], time.time()
for i, fname in enumerate(raw_photos, 1):
    stem = os.path.splitext(fname)[0]
    crop_name = f"{stem}_crop.jpg"
    out_path = os.path.join(CROPS_DIR, crop_name)
    if os.path.exists(out_path):
        records.append({"original_filename": fname, "crop_filename": crop_name, "method": "cached"})
        continue
    img = cv2.imread(os.path.join(KIGALI_IMAGES_DIR, fname))
    if img is None:
        records.append({"original_filename": fname, "crop_filename": "", "method": "load_error"})
        continue
    crop, method = wp.crop_display(img, yolo, device=yolo_device)
    if crop is None:
        records.append({"original_filename": fname, "crop_filename": "", "method": method})
        continue
    cv2.imwrite(out_path, crop)
    records.append({"original_filename": fname, "crop_filename": crop_name, "method": method})
    if i % 100 == 0:
        print(f"  {i}/{len(raw_photos)} photos processed")

crops_meta = pd.DataFrame(records)
crops_meta.to_csv(os.path.join(RESULTS_DIR, "kigali_crops_meta.csv"), index=False)
print(f"\nCropping finished in {time.time() - t0:.0f}s")
print(crops_meta["method"].value_counts().to_string())

on_disk = set(os.listdir(CROPS_DIR))
miss_lab = ~kigali_df["crop_filename"].isin(on_disk)
miss_unr = ~unread_df["crop_filename"].isin(on_disk)
print(f"\nLabelled crops found:   {(~miss_lab).sum()} / {len(kigali_df)}")
print(f"Unreadable crops found: {(~miss_unr).sum()} / {len(unread_df)}")
if miss_lab.any():
    print("Missing labelled crops (removed):", kigali_df.loc[miss_lab, "crop_filename"].tolist()[:10])
kigali_df = kigali_df[~miss_lab].reset_index(drop=True)
unread_df = unread_df[~miss_unr].reset_index(drop=True)

## 3.3 Rebuild the Yandex labels from the file names

**The problem we found.** The Yandex CSV stores each reading as a decimal number, for example
`21.86` cubic metres. Turning that number into a digit string caused two silent errors:

1. **Lost digits.** The meter shows `00021860`: five black wheels (cubic metres) and three red wheels
   (litres). The number `21.86` has lost the two leading zeros *and* the trailing zero of the litres.
2. **Floating-point noise.** Computers store decimals approximately, so a value like `90.216` can turn
   into `9021600000000001` when converted carelessly.

The labels then did not match what is printed on the meter, and the model could not learn. Training
collapsed into predicting nothing at all (the "CTC blank trap"). An overfit test on 32 images proved
the code was fine and the labels were wrong.

**The fix.** Each Yandex file name contains the exact reading as text, e.g. `id_53_value_595_825.jpg`.
We rebuild every label from the file name as the full 8-wheel display:

    integer part '595' → padded to 5 digits → '00595'
    decimal part '825' → padded to 3 digits → '825'
    label = '00595825'

In [ ]:
def label_from_filename(name):
    # 'id_53_value_595_825.jpg' -> '00595825' (5 cubic-metre digits + 3 litre digits).
    m = re.search(r"value_(\d+)(?:_(\d+))?", str(name))
    if not m:
        return None
    integer, decimal = m.group(1), (m.group(2) or "")
    label = integer.zfill(5) + decimal.ljust(3, "0")[:3]
    return label if len(label) == 8 else None

def naive_label(value):
    # The fragile conversion that caused the problem, shown for comparison only.
    return str(value).replace(".", "")

yandex_raw["label"] = yandex_raw["photo_name"].map(label_from_filename)
yandex_raw["naive_label"] = yandex_raw["value"].map(naive_label)

print("Examples (naive conversion vs label rebuilt from file name):")
print(yandex_raw[["photo_name", "value", "naive_label", "label"]].head(8).to_string(index=False))
bad = yandex_raw["label"].isna()
print(f"\nLabels rebuilt: {(~bad).sum()} / {len(yandex_raw)} (dropped {bad.sum()} that are not 8 digits)")
print(f"Naive labels that differ from the true display: "
      f"{(yandex_raw['naive_label'] != yandex_raw['label']).sum()}")

## 3.4 Cut out the number window from every Yandex photo

Yandex photos come with a hand-drawn **polygon** around the number window. We rotate the photo so the
polygon's first edge lies flat, then crop it. If the crop comes out taller than it is wide, the digits
are standing on end, so we turn it 90°.

Some crops will still be **upside down**, because the polygon's first point is not always on the same
corner. We fix that in Section 9.2 once we have a model that can tell which way up the digits are.

In [ ]:
def parse_polygon(location_str, w, h):
    loc = ast.literal_eval(location_str)
    return [(p["x"] * w, p["y"] * h) for p in loc["data"]]

def oriented_crop(img_rgb, points):
    pts = np.array(points, dtype=np.float32)
    angle = math.degrees(math.atan2(pts[1][1] - pts[0][1], pts[1][0] - pts[0][0]))
    center = tuple(map(float, pts.mean(axis=0)))
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    h, w = img_rgb.shape[:2]
    diag = int(math.hypot(w, h))
    M[0, 2] += (diag - w) / 2
    M[1, 2] += (diag - h) / 2
    rotated = cv2.warpAffine(img_rgb, M, (diag, diag), borderMode=cv2.BORDER_REPLICATE)
    new_pts = (M @ np.hstack([pts, np.ones((len(pts), 1), np.float32)]).T).T
    x1, y1 = np.maximum(new_pts.min(axis=0).astype(int), 0)
    x2, y2 = new_pts.max(axis=0).astype(int)
    return rotated[y1:y2, x1:x2]

rows = []
for idx, r in yandex_raw.iterrows():
    if r["label"] is None or pd.isna(r["label"]):
        continue
    name = f"yandex_crop_{idx}.jpg"
    out = os.path.join(YANDEX_CROPS, name)
    if not os.path.exists(out):
        try:
            img = np.array(Image.open(os.path.join(YANDEX_IMAGES_DIR, r["photo_name"])).convert("RGB"))
            crop = oriented_crop(img, parse_polygon(r["location"], img.shape[1], img.shape[0]))
            if crop.size == 0:
                continue
            if crop.shape[0] > crop.shape[1]:
                crop = cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE)
            cv2.imwrite(out, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        except Exception as e:
            print(f"  skipped {r['photo_name']}: {e}")
            continue
    rows.append({"crop_filename": name, "photo_name": r["photo_name"], "label": r["label"]})

yandex_df = pd.DataFrame(rows)
yandex_df.to_csv(os.path.join(RESULTS_DIR, "yandex_labels_fixed.csv"), index=False)
print(f"Yandex crops ready: {len(yandex_df)} / {len(yandex_raw)}")
yandex_df.head()

---
# 4. Exploratory data analysis (EDA)

EDA means looking closely at the data before modelling, to find problems early and to make informed
choices. Each check below led to a concrete decision later in the notebook.

## 4.1 Label lengths

A WASAC meter has 8 wheels, so a perfect label has 8 digits. Shorter labels usually mean leading zeros
were left out; very short ones are partial crops.

In [ ]:
kigali_df["raw_len"] = kigali_df["label_raw"].str.len()
yandex_df["len"] = yandex_df["label"].str.len()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
kigali_df["raw_len"].value_counts().sort_index().plot.bar(ax=ax[0], color="#1f4e79")
ax[0].set_title("Kigali label length (digits)"); ax[0].set_xlabel("digits"); ax[0].set_ylabel("labels")
kigali_df["verification_status"].value_counts().plot.bar(ax=ax[1], color="#4a7ab5")
ax[1].set_title("Kigali verification status"); ax[1].tick_params(axis="x", rotation=15)
yandex_df["len"].value_counts().sort_index().plot.bar(ax=ax[2], color="#7a9cc6")
ax[2].set_title("Yandex label length after rebuild")
plt.tight_layout(); save_plot("eda_label_lengths.png"); plt.show()

print(kigali_df["raw_len"].describe().round(2).to_string())

## 4.2 A hidden problem: leading zeros were lost

The meter always shows its leading zeros (`00565846`). So the share of labels starting with `0` should be
high for **every** verification status. The table below shows whether that is true.

If `VERIFIED_CORRECT` labels almost never start with 0 while `VERIFIED_CORRECTED` labels usually do, the
zeros were lost somewhere, most likely when the verification sheet was saved as numbers (a spreadsheet
silently turns `00565846` into `565846`). The labels I typed myself kept their zeros.

**Decision:** this inconsistency would confuse the model (the same picture would have two different
"correct" answers), so in Section 9.3 the missing zeros are restored in a careful, tested way. And because
leading zeros never change a bill, **reading accuracy, which ignores them, is the headline metric.**

In [ ]:
zero_share = (kigali_df.assign(starts_with_0=kigali_df["label_raw"].str.startswith("0"))
              .groupby("verification_status")["starts_with_0"].agg(["mean", "count"]))
zero_share["mean"] = (100 * zero_share["mean"]).round(1)
zero_share.columns = ["% starting with 0", "labels"]
print(zero_share.to_string())

## 4.3 Image size and sharpness

A common idea is that unreadable photos are simply blurry. We test it with the **Laplacian variance**,
a standard sharpness score: it measures how strong the edges in an image are. Blurry images score low.

If readable and unreadable crops have similar sharpness, then **blur alone cannot tell them apart**, and
the quality gate needs a learned model (MobileNetV2) rather than a single threshold. This is what
justifies Stage 1.

In [ ]:
def image_stats(fname):
    g = cv2.imread(os.path.join(CROPS_DIR, fname), cv2.IMREAD_GRAYSCALE)
    if g is None:
        return pd.Series({"width": np.nan, "height": np.nan, "sharpness": np.nan})
    return pd.Series({"width": g.shape[1], "height": g.shape[0],
                      "sharpness": cv2.Laplacian(g, cv2.CV_64F).var()})

kigali_df[["width", "height", "sharpness"]] = kigali_df["crop_filename"].apply(image_stats)
unread_df[["width", "height", "sharpness"]] = unread_df["crop_filename"].apply(image_stats)
kigali_df["aspect"] = kigali_df["width"] / kigali_df["height"]

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(kigali_df["width"], bins=30, color="#1f4e79"); ax[0].set_title("Crop width (px)")
ax[1].hist(kigali_df["aspect"], bins=30, color="#4a7ab5"); ax[1].set_title("Crop width / height")
ax[2].boxplot([np.log10(kigali_df["sharpness"] + 1), np.log10(unread_df["sharpness"] + 1)])
ax[2].set_xticks([1, 2]); ax[2].set_xticklabels(["readable", "unreadable"])
ax[2].set_title("Sharpness (log10 Laplacian variance)")
plt.tight_layout(); save_plot("eda_image_stats.png"); plt.show()

print(f"Median crop size: {kigali_df['width'].median():.0f} x {kigali_df['height'].median():.0f} px")
print(f"Median sharpness  readable: {kigali_df['sharpness'].median():.0f} | "
      f"unreadable: {unread_df['sharpness'].median():.0f}")

## 4.4 Why are some photos unreadable?

These are the reasons I recorded while verifying. They show that "unreadable" is mostly about dirt,
moisture and wheels caught between two digits, not about blur.

In [ ]:
reasons = unread_df["reason"].str.lower().str.replace(r" - .*", "", regex=True).str.strip()
print(reasons.value_counts().to_string())

## 4.5 The same meter appears many times

While verifying, I noticed the same meters were photographed several times. If the same meter ends up
in both the training set and the test set, the model can **remember** that meter instead of **reading**
it, and the test score becomes too optimistic. This is called **data leakage**.

Two photos showing the same reading (or readings that differ only in the last two digits, i.e. taken
on different days) almost certainly show the same meter. The count below shows how common this is.

**Decision:** split the data **by meter**, not by photo (Section 5).

In [ ]:
reading_counts = kigali_df["label_raw"].map(strip0).value_counts()
repeated = reading_counts[reading_counts > 1]
print(f"Distinct readings: {len(reading_counts)} for {len(kigali_df)} photos")
print(f"Readings that appear more than once: {len(repeated)}, covering {repeated.sum()} photos "
      f"({100 * repeated.sum() / len(kigali_df):.0f}%)")
print("Most repeated:", repeated.head(8).to_dict())

## 4.6 What the crops look like

Top two rows: readable Kigali crops with their labels. Third row: unreadable crops. Bottom row: Yandex
crops. Notice the visual difference between Yandex and Kigali meters (the domain gap), and that some
Yandex crops are upside down.

In [ ]:
def show(ax, path, title):
    img = cv2.imread(path)
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(4, 4, figsize=(16, 7))
samp = kigali_df.sample(min(8, len(kigali_df)), random_state=SEED)
for ax, (_, r) in zip(axes[:2].ravel(), samp.iterrows()):
    show(ax, os.path.join(CROPS_DIR, r["crop_filename"]), f"{r['label_raw']} ({r['verification_status'][9:]})")
for ax, (_, r) in zip(axes[2], unread_df.sample(min(4, len(unread_df)), random_state=SEED).iterrows()):
    show(ax, os.path.join(CROPS_DIR, r["crop_filename"]), "unreadable")
for ax, (_, r) in zip(axes[3], yandex_df.sample(min(4, len(yandex_df)), random_state=SEED).iterrows()):
    show(ax, os.path.join(YANDEX_CROPS, r["crop_filename"]), f"Yandex {r['label']}")
plt.tight_layout(); save_plot("eda_samples.png"); plt.show()

---
# 5. Data splitting

We divide the data into three sets, each with one job:

| Set | Share | Job |
|---|---|---|
| **Train** | 70% | The model learns from these |
| **Validation** | 15% | We compare models, choose settings and pick the best epoch |
| **Test** | 15% | Locked away. Used **once**, at the very end, to report the final result |

The test set must stay untouched because every decision made while looking at a set makes that set's
score a little too optimistic. Only a set that influenced nothing gives an honest estimate.

## 5.1 Group photos by meter

To prevent leakage (Section 4.5), photos are grouped so that **all photos of one meter stay together**.
Two photos are put in the same group if:

- their readings are identical (ignoring leading zeros), or
- their readings are identical except for the last two digits (same meter, a few litres later).

Groups are chained: if A matches B and B matches C, all three are one meter. This is done with a
**union-find** structure. The rule is deliberately strict: it may occasionally put two different meters
in one group, which makes the test *harder*, never easier.

In [ ]:
def group_meters(labels):
    parent = list(range(len(labels)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    owner = {}
    for i, lab in enumerate(labels):
        s = strip0(lab)
        keys = ["E:" + s] + (["M:" + s[:-2]] if len(s) >= 5 else [])
        for k in keys:
            if k in owner:
                parent[find(i)] = find(owner[k])
            else:
                owner[k] = i
    return [find(i) for i in range(len(labels))]

kigali_df["meter_group"] = group_meters(kigali_df["label_raw"].tolist())
sizes = kigali_df["meter_group"].value_counts()
print(f"Photos: {len(kigali_df)} | distinct meters: {len(sizes)}")
print(f"Photos per meter: median {sizes.median():.0f}, max {sizes.max()}, meters with one photo: {(sizes == 1).sum()}")

## 5.2 Split Kigali by meter (70 / 15 / 15)

`GroupShuffleSplit` splits **groups**, not rows, so every meter lands in exactly one set. The code then
**proves** there is no leakage with checks that stop the notebook if any meter is shared.

One more step, for the training set only: a meter photographed 14 times would otherwise teach the model
far more about that one meter than about the others. So training keeps **at most 4 photos per meter**.
Validation and test keep every photo, because they measure performance and we want all the evidence.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trval_idx, test_idx = next(gss.split(kigali_df, groups=kigali_df["meter_group"]))
trval, test_df = kigali_df.iloc[trval_idx].reset_index(drop=True), kigali_df.iloc[test_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=SEED)
tr_idx, va_idx = next(gss2.split(trval, groups=trval["meter_group"]))
train_full, val_df = trval.iloc[tr_idx].reset_index(drop=True), trval.iloc[va_idx].reset_index(drop=True)

def meter_keys(frame):
    keys = set()
    for lab in frame["label_raw"]:
        s = strip0(lab); keys.add("E:" + s)
        if len(s) >= 5:
            keys.add("M:" + s[:-2])
    return keys

for a, b, name in [(train_full, val_df, "train/val"), (train_full, test_df, "train/test"), (val_df, test_df, "val/test")]:
    assert not set(a.meter_group) & set(b.meter_group), f"Meter shared between {name}"
    assert not meter_keys(a) & meter_keys(b), f"Reading shared between {name}"
print("Leakage checks passed: no meter appears in more than one set.")

TRAIN_CAP = 4
train_df = (train_full.sample(frac=1, random_state=SEED)      # shuffle, then keep the first 4 per meter
            .groupby("meter_group").head(TRAIN_CAP)
            .reset_index(drop=True))

summary = pd.DataFrame({
    "photos": [len(train_df), len(val_df), len(test_df)],
    "meters": [train_df.meter_group.nunique(), val_df.meter_group.nunique(), test_df.meter_group.nunique()],
}, index=["train (capped)", "validation", "test"])
print(f"Training photos before the cap: {len(train_full)} -> after: {len(train_df)}")
print(summary.to_string())
print("\nVerification status per set:")
print(pd.DataFrame({"train": train_df.verification_status.value_counts(),
                    "val": val_df.verification_status.value_counts(),
                    "test": test_df.verification_status.value_counts()}).fillna(0).astype(int).to_string())

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    frame.to_csv(os.path.join(SPLITS_DIR, f"kigali_{name}.csv"), index=False)

## 5.3 Yandex and unreadable splits

- **Yandex (90 / 10):** only used for pretraining, so it needs a training set and a small validation set
  to watch progress. It never touches the Kigali test.
- **Unreadable crops (70 / 15 / 15):** for the quality gate. These crops have no reading, so they cannot
  be grouped by meter; they are split at random. This is a limitation noted in the report.

In [ ]:
y_train, y_val = train_test_split(yandex_df, test_size=0.10, random_state=SEED)
y_train, y_val = y_train.reset_index(drop=True), y_val.reset_index(drop=True)

u_train, u_tmp = train_test_split(unread_df, test_size=0.30, random_state=SEED)
u_val, u_test = train_test_split(u_tmp, test_size=0.50, random_state=SEED)
u_train, u_val, u_test = [d.reset_index(drop=True) for d in (u_train, u_val, u_test)]

for name, frame in [("yandex_train", y_train), ("yandex_val", y_val),
                    ("unreadable_train", u_train), ("unreadable_val", u_val), ("unreadable_test", u_test)]:
    frame.to_csv(os.path.join(SPLITS_DIR, f"{name}.csv"), index=False)
print(f"Yandex:     {len(y_train)} train | {len(y_val)} val")
print(f"Unreadable: {len(u_train)} train | {len(u_val)} val | {len(u_test)} test")

---
# 6. Preprocessing and feature engineering

Deep learning models learn their own features from pixels, so "feature engineering" here means
**preparing the images well** and **augmentation**.

## 6.1 Turning a crop into model input

Every crop goes through the same steps:

1. **Grayscale.** Colour is not needed to read digits, and it removes differences between meter brands.
2. **Resize to 64 × 256 pixels.** The model needs a fixed size. The 1:4 shape matches a long, thin number
   window.
3. **Scale pixel values** from 0–255 to roughly −1 to 1, which helps training stay stable.

## 6.2 Augmentation (training only)

Augmentation creates slightly changed copies of training images each epoch, so the model sees more
variety and memorises less. Each change imitates something that really happens to phone photos:

| Change | Imitates |
|---|---|
| Brightness and contrast | Shade, sunlight, dirty glass |
| Small rotation, shift and zoom-out | A slightly tilted phone; zoom-out also teaches the model to read digits near the edges, which reduces dropped digits |
| Light blur | Slight camera shake |

**What we deliberately do *not* use:** random erasing (blacking out rectangles). It can hide a whole digit
while the label still contains it, which teaches the model to invent digits. This was one of the three
causes of the early training collapse.

## 6.3 How labels become targets

The model outputs one of 11 symbols at each of 32 positions: the digits 0–9 and a special **blank**
(symbol 10). The **CTC loss** (Connectionist Temporal Classification) lets the model learn from a label
like `00565846` without being told *where* each digit is in the image. When decoding, repeated symbols
are merged and blanks removed:

    0 0 _ 0 5 _ 6 6 _ 5 ...  →  0 0 5 6 5 ...   (a blank separates two real zeros)

In [ ]:
IMG_H, IMG_W = 64, 256
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
train_tf = transforms.Compose([
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomApply([transforms.RandomAffine(degrees=3, translate=(0.03, 0.05),
                                                    scale=(0.9, 1.05), fill=0)], p=0.6),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),
    eval_tf,
])

class MeterDataset(Dataset):
    # Returns (image tensor, label string). Works for Kigali and Yandex crops.
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), [str(l) for l in labels], tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

def ctc_collate(batch):
    imgs, labels = zip(*batch)
    targets = torch.tensor([int(c) for l in labels for c in l], dtype=torch.long)
    lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return torch.stack(imgs), targets, lengths, list(labels)

def kigali_paths(frame):
    return [os.path.join(CROPS_DIR, f) for f in frame["crop_filename"]]

def make_loader(paths, labels, tf, batch_size=32, shuffle=False, sampler=None):
    return DataLoader(MeterDataset(paths, labels, tf), batch_size=batch_size,
                      shuffle=shuffle if sampler is None else False, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=PIN, collate_fn=ctc_collate)

# Preview: one crop and four augmented versions of it
example = Image.open(kigali_paths(train_df.head(1))[0]).convert("RGB")
fig, axes = plt.subplots(1, 5, figsize=(18, 2.2))
axes[0].imshow(eval_tf(example)[0], cmap="gray"); axes[0].set_title("model input")
for ax in axes[1:]:
    ax.imshow(train_tf(example)[0], cmap="gray"); ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); save_plot("preprocessing_augmentation.png"); plt.show()

---
# 7. Baselines

A baseline is a simple method any new model must clearly beat. Without one, "64% accuracy" means
nothing: is that good or bad? Baselines are measured on the **validation** set here; the final test
comparison is in Section 12.

## 7.1 Baseline for reading: EasyOCR

EasyOCR is a popular free text-recognition library that works out of the box. If it already read meters
well, a custom model would not be needed.

In [ ]:
def easyocr_predict(files):
    try:
        import easyocr
        reader = easyocr.Reader(["en"], gpu=DEVICE.type == "cuda", verbose=False)
    except Exception as e:
        print("EasyOCR unavailable:", e)
        return None
    preds = []
    for f in files:
        try:
            preds.append(digits_only("".join(reader.readtext(f, allowlist="0123456789", detail=0))))
        except Exception:
            preds.append("")
    return preds

easyocr_val = easyocr_predict(kigali_paths(val_df))
if easyocr_val is not None:
    m = reading_metrics(easyocr_val, val_df["label_raw"], val_df["meter_group"])
    print("EasyOCR on Kigali validation:", fmt(m))
    for p, g in list(zip(easyocr_val, val_df["label_raw"]))[:6]:
        print(f"   label {g:>10}   EasyOCR {p:>16}")

## 7.2 Baseline for the quality gate: a blur threshold

The simplest quality gate is a rule: *"if the sharpness score is below X, ask for a retake."* We pick the
threshold X that works best on the training data and measure it on validation. The unreadable class is
treated as **positive** (the thing we want to catch).

In [ ]:
def sharpness_of(files):
    out = []
    for f in files:
        g = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        out.append(cv2.Laplacian(g, cv2.CV_64F).var() if g is not None else 0.0)
    return np.array(out)

def qg_frame(readable, unreadable):
    return pd.concat([readable[["crop_filename"]].assign(y=0), unreadable[["crop_filename"]].assign(y=1)],
                     ignore_index=True)

qg_train = qg_frame(train_df, u_train)
qg_val = qg_frame(val_df, u_val)
qg_test = qg_frame(test_df, u_test)   # readable rows first, in test_df order

lap_train = sharpness_of([os.path.join(CROPS_DIR, f) for f in qg_train.crop_filename])
cands = np.percentile(lap_train, np.arange(1, 100))
LAP_THRESHOLD = cands[int(np.argmax([f1_score(qg_train.y, lap_train < c, zero_division=0) for c in cands]))]

lap_val = sharpness_of([os.path.join(CROPS_DIR, f) for f in qg_val.crop_filename])
pred = (lap_val < LAP_THRESHOLD).astype(int)
print(f"Blur threshold chosen on train: {LAP_THRESHOLD:.1f}")
print(f"Validation: precision {100*precision_score(qg_val.y, pred, zero_division=0):.1f}% | "
      f"recall {100*recall_score(qg_val.y, pred, zero_division=0):.1f}% | "
      f"F1 {100*f1_score(qg_val.y, pred, zero_division=0):.1f}% | "
      f"AUC {roc_auc_score(qg_val.y, -lap_val):.3f}")

---
# 8. Model architecture

## 8.1 Main model: CRNN with CTC

A **CRNN** (Convolutional Recurrent Neural Network, Shi et al., 2016) is the standard architecture for
reading a line of text from an image. It has three parts:

1. **CNN (the eyes).** Convolution layers scan the 64 × 256 image and shrink it to a row of
   **32 feature columns**, each describing a thin vertical slice of the number window.
2. **BiLSTM (the context).** A bidirectional LSTM reads the 32 columns left-to-right *and*
   right-to-left, so each column "knows" its neighbours. This helps with wheels caught between two digits.
3. **Output layer + CTC.** Each of the 32 columns predicts one of 11 symbols (0–9 or blank). CTC decoding
   turns these into the reading.

**Why 32 columns?** An earlier version produced 64 columns for 8 digits. With so many positions, the
easiest way for the model to lower its loss early in training was to predict "blank" everywhere, and it
got stuck there. 32 columns still leave room for 8 digits plus the blanks needed between repeated digits
(`00000072`), while making the collapse much less likely.

## 8.2 Alternative model: ResNet18 + BiGRU

The workflow requires comparing against a different architecture. The alternative replaces the small
CNN with **ResNet18**, a deeper network pre-trained on ImageNet (1.2 million everyday photos), and the
LSTM with a **GRU** (a simpler recurrent layer). One ResNet layer is changed to shrink only the height,
so it also outputs 32 columns and the comparison is fair.

## 8.3 Shared model code (`wasac_models.py`)

As with preprocessing, the model code is written to a file that the backend will use, so the trained
weights always load into exactly the same architecture.

In [ ]:
%%writefile /kaggle/working/deploy/wasac_models.py
"""WASAC meter reading models, shared by the training notebook and the FastAPI backend."""
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image

from wasac_preprocessing import pad_reading, digits_to_m3

BLANK = 10            # CTC blank symbol; digits are 0..9
NUM_CLASSES = 11
IMG_H, IMG_W = 64, 256


def eval_transform():
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_H, IMG_W)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ])


def _conv(cin, cout):
    return [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]


class CRNN(nn.Module):
    """CNN feature extractor -> 2-layer BiLSTM -> per-column symbol scores (32 columns)."""

    def __init__(self, num_classes=NUM_CLASSES, hidden=256):
        super().__init__()
        self.cnn = nn.Sequential(
            *_conv(1, 64), nn.MaxPool2d(2, 2),            # 64x256 -> 32x128
            *_conv(64, 128), nn.MaxPool2d(2, 2),          # -> 16x64
            *_conv(128, 256), *_conv(256, 256),
            nn.MaxPool2d(2, 2),                           # -> 8x32
            *_conv(256, 256), nn.Dropout2d(0.1),
            nn.MaxPool2d((2, 1)),                         # -> 4x32 (height only)
            nn.AdaptiveAvgPool2d((1, None)),              # -> 1x32
        )
        self.rnn = nn.LSTM(256, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        features = self.cnn(x).squeeze(2).permute(0, 2, 1)   # (batch, 32, 256)
        out, _ = self.rnn(features)
        return self.fc(out).log_softmax(-1)                  # (batch, 32, 11)


class ResNetGRU(nn.Module):
    """ResNet18 (layers 1-3) -> 2-layer BiGRU -> per-column symbol scores (32 columns)."""

    def __init__(self, num_classes=NUM_CLASSES, hidden=256, imagenet=False):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if imagenet else None
        r = torchvision.models.resnet18(weights=weights)
        r.layer3[0].conv1.stride = (2, 1)            # shrink height only -> 32 columns
        r.layer3[0].downsample[0].stride = (2, 1)
        self.backbone = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3)
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.GRU(256, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)                 # grayscale -> 3 channels for ResNet
        features = self.pool(self.backbone(x)).squeeze(2).permute(0, 2, 1)
        out, _ = self.rnn(features)
        return self.fc(out).log_softmax(-1)


def build_model(arch, imagenet=False):
    if arch == "crnn":
        return CRNN()
    if arch == "resnet_gru":
        return ResNetGRU(imagenet=imagenet)
    raise ValueError(f"Unknown architecture: {arch}")


def greedy_decode(log_probs):
    """CTC greedy decoding: best symbol per column, merge repeats, drop blanks."""
    texts = []
    for seq in log_probs.argmax(-1).cpu().numpy():
        out, prev = [], None
        for p in seq:
            if p != prev and p != BLANK:
                out.append(str(int(p)))
            prev = p
        texts.append("".join(out))
    return texts


def decode_with_confidence(log_probs):
    """Like greedy_decode, plus the mean and the lowest probability of the emitted digits."""
    probs = log_probs.exp()
    best_p, best_i = probs.max(-1)
    results = []
    for ps, ids in zip(best_p.cpu().numpy(), best_i.cpu().numpy()):
        out, conf, prev = [], [], None
        for p, i in zip(ps, ids):
            if i != prev and i != BLANK:
                out.append(str(int(i)))
                conf.append(float(p))
            prev = i
        results.append(("".join(out), sum(conf) / len(conf) if conf else 0.0, min(conf) if conf else 0.0))
    return results


class MeterReader:
    """Loads trained weights and reads a CROPPED meter display (see wasac_preprocessing.crop_display)."""

    def __init__(self, weights_path, arch="crnn", device="cpu"):
        self.device = torch.device(device)
        self.model = build_model(arch).to(self.device)
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.eval()
        self.tf = eval_transform()

    @torch.no_grad()
    def read(self, image):
        if not isinstance(image, Image.Image):
            image = Image.open(image)
        x = self.tf(image.convert("RGB")).unsqueeze(0).to(self.device)
        raw, conf_mean, conf_min = decode_with_confidence(self.model(x))[0]
        padded = pad_reading(raw)
        return {
            "raw_digits": raw,                 # exactly what the model read
            "reading_digits": padded,          # 8 digits, or None if the length is impossible
            "reading_m3": digits_to_m3(padded),
            "confidence_mean": round(conf_mean, 4),
            "confidence_min": round(conf_min, 4),
            "needs_check": padded is None,
        }

In [ ]:
import wasac_models as wm
importlib.reload(wm)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

dummy = torch.zeros(2, 1, IMG_H, IMG_W)
for arch in ["crnn", "resnet_gru"]:
    m = wm.build_model(arch)
    out = m(dummy)
    print(f"{arch:11s} parameters: {n_params(m):>10,} | output shape: {tuple(out.shape)} (batch, columns, symbols)")
    assert out.shape[1] == 32, "Both models must output 32 columns"
del m, out

---
# 9. Training

## 9.1 The training loop

One function trains every reading model in this notebook, so all experiments use the same rules:

- **Optimiser:** AdamW (Adam with weight decay, which discourages very large weights).
- **Learning rate schedule:** a short warm-up (the rate rises over the first epoch), then a smooth
  *cosine* decrease. Starting gently helps CTC training avoid the blank trap.
- **Gradient clipping at 5:** stops a single bad batch from making a huge, destructive update.
- **Model checkpointing:** after every epoch the model is scored on the validation set, and the best
  version is kept, not the last one.
- **Early stopping (optional):** stop if validation has not improved for a number of epochs.
- **Freezing (fine-tuning only):** for the first few epochs the CNN part is frozen (not updated), so the
  new data first adapts the upper layers without destroying what the CNN learned. After that, the CNN is
  unfrozen with a smaller learning rate than the rest.

In [ ]:
CTC_LOSS = nn.CTCLoss(blank=wm.BLANK, zero_infinity=True)

def ctc_loss(model, imgs, targets, lengths):
    log_probs = model(imgs.to(DEVICE, non_blocking=True))
    steps = torch.full((log_probs.size(0),), log_probs.size(1), dtype=torch.long)
    return CTC_LOSS(log_probs.permute(1, 0, 2).float(), targets, steps, lengths)

@torch.no_grad()
def mean_ctc_loss(model, loader):
    # Average CTC loss over a loader in eval mode (no augmentation, no dropout): the validation loss.
    model.eval()
    total, n = 0.0, 0
    for imgs, targets, lengths, labels in loader:
        total += ctc_loss(model, imgs, targets, lengths).item() * len(labels)
        n += len(labels)
    return total / max(1, n)

def eval_subset_loader(paths, labels, n=200, seed=SEED):
    # A fixed, non-augmented sample of the TRAINING data, scored every epoch for the learning curves.
    idx = np.random.default_rng(seed).permutation(len(paths))[:n]
    return make_loader([paths[i] for i in idx], [list(labels)[i] for i in idx], eval_tf)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, gts = [], []
    for imgs, _, _, labels in loader:
        preds += wm.greedy_decode(model(imgs.to(DEVICE)))
        gts += list(labels)
    return preds, gts

def train_ctc(model, train_loader, val_loader, *, epochs, lr, name, select="reading %",
              backbone_prefix=None, backbone_lr=None, freeze_epochs=0, patience=None, val_groups=None,
              train_eval_loader=None):
    model.to(DEVICE)
    if backbone_prefix:
        bb = [p for n, p in model.named_parameters() if n.startswith(backbone_prefix)]
        rest = [p for n, p in model.named_parameters() if not n.startswith(backbone_prefix)]
        assert bb, f"No parameters start with '{backbone_prefix}'"
        groups = [{"params": bb, "lr": backbone_lr}, {"params": rest, "lr": lr}]
    else:
        groups = [{"params": list(model.parameters()), "lr": lr}]
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    total, warm = epochs * len(train_loader), max(1, len(train_loader))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm
        else max(0.02, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))

    history, best_state, best_key, bad, t0 = [], None, None, 0, time.time()
    for epoch in range(1, epochs + 1):
        frozen = bool(backbone_prefix) and epoch <= freeze_epochs
        if backbone_prefix:
            for n, p in model.named_parameters():
                if n.startswith(backbone_prefix):
                    p.requires_grad = not frozen
        model.train()
        losses = []
        for imgs, targets, lengths, _ in train_loader:
            loss = ctc_loss(model, imgs, targets, lengths)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step()
            losses.append(loss.item())
        preds, gts = predict(model, val_loader)
        m = reading_metrics(preds, gts, val_groups)
        key = (m[select], -m["edit dist"])
        improved = best_key is None or key > best_key
        if improved:
            best_key, best_state, bad = key, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        # Learning-curve values: validation loss every epoch, and accuracy on a fixed slice of the
        # training data (same metric as validation), so the gap between the two shows over-fitting.
        row = {"epoch": epoch, "loss": float(np.mean(losses)), "val loss": mean_ctc_loss(model, val_loader), **m}
        if train_eval_loader is not None:
            tm = reading_metrics(*predict(model, train_eval_loader))
            row["train reading %"], row["train exact %"] = tm["reading %"], tm["exact %"]
        history.append(row)
        print(f"{name} | ep {epoch:02d} | loss {np.mean(losses):.3f} | val loss {row['val loss']:.3f} | exact {m['exact %']:5.1f} | "
              f"reading {m['reading %']:5.1f} | dist {m['edit dist']:.2f}"
              f"{' | frozen' if frozen else ''}{'  *' if improved else ''}")
        if patience and bad >= patience:
            print(f"{name} | early stop at epoch {epoch}")
            break
    for p in model.parameters():
        p.requires_grad = True
    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(MODELS_DIR, f"{name}.pt"))
    print(f"{name} | best {select} {best_key[0]:.1f} | {(time.time() - t0) / 60:.1f} min | saved {name}.pt")
    return model, pd.DataFrame(history)

def plot_history(hist, title, filename, metric="reading %", axes=None):
    # Learning curves: left, training vs validation CTC loss; right, training vs validation accuracy.
    # The dashed line marks the epoch that was kept (best validation score).
    own = axes is None
    if own:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    best = hist.loc[hist[metric].idxmax(), "epoch"]
    axes[0].plot(hist["epoch"], hist["loss"], label="training loss")
    if "val loss" in hist:
        axes[0].plot(hist["epoch"], hist["val loss"], label="validation loss")
    axes[0].set_title(f"{title}: CTC loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
    if f"train {metric}" in hist:
        axes[1].plot(hist["epoch"], hist[f"train {metric}"], label=f"training {metric}")
    axes[1].plot(hist["epoch"], hist[metric], label=f"validation {metric}")
    axes[1].set_title(f"{title}: accuracy"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("%")
    axes[1].set_ylim(0, 100)
    for a in axes:
        a.axvline(best, ls="--", c="grey", lw=1, label=f"kept: epoch {best}")
        a.legend(fontsize=8); a.grid(alpha=0.3)
    if own:
        plt.tight_layout(); save_plot(filename); plt.show()

## 9.2 Pretraining on Yandex

### Round 1

The CRNN starts from random weights and learns to read the 1,119 Yandex training crops. Here every digit
matters (all Yandex labels are exactly 8 digits), so the best epoch is chosen by **exact match**.

Expect the first epochs to predict nothing or only zeros; that is normal for CTC. Accuracy usually jumps
once the model "finds" the digits.

In [ ]:
seed_everything()
y_train_loader = make_loader([os.path.join(YANDEX_CROPS, f) for f in y_train.crop_filename], y_train.label,
                             train_tf, batch_size=32, shuffle=True)
y_val_loader = make_loader([os.path.join(YANDEX_CROPS, f) for f in y_val.crop_filename], y_val.label, eval_tf)

y_train_eval = eval_subset_loader([os.path.join(YANDEX_CROPS, f) for f in y_train.crop_filename], y_train.label)
crnn_r1, hist_r1 = train_ctc(wm.CRNN(), y_train_loader, y_val_loader, epochs=EPOCHS["yandex_pretrain"],
                             lr=1e-3, name="crnn_yandex_round1", select="exact %", train_eval_loader=y_train_eval)
plot_history(hist_r1, "Yandex round 1", "train_yandex_round1.png", metric="exact %")

### Fixing upside-down crops

Some Yandex crops are upside down (Section 3.4). The round-1 model already reads well, so we can use it
as a judge: for each crop we compare the original and a 180°-rotated copy, and keep the version where the
**known label** is most likely according to the model (lowest CTC loss).

**Honesty note:** this step uses the labels, including those of the Yandex validation crops. That makes
the Yandex validation score slightly optimistic. It does not affect the Kigali results, and the Yandex
score is only reported as a pretraining result, never as the headline.

In [ ]:
@torch.no_grad()
def per_sample_ctc(model, imgs, labels):
    log_probs = model(imgs.to(DEVICE))
    targets = torch.tensor([int(c) for l in labels for c in l], dtype=torch.long)
    lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    steps = torch.full((len(labels),), log_probs.size(1), dtype=torch.long)
    return F.ctc_loss(log_probs.permute(1, 0, 2).float(), targets, steps, lengths, blank=wm.BLANK,
                      reduction="none", zero_infinity=True).cpu().numpy()

crnn_r1.eval()
flipped, B = [], 64
for start in range(0, len(yandex_df), B):
    chunk = yandex_df.iloc[start:start + B]
    imgs = [Image.open(os.path.join(YANDEX_CROPS, f)).convert("RGB") for f in chunk.crop_filename]
    labels = chunk.label.tolist()
    loss_0 = per_sample_ctc(crnn_r1, torch.stack([eval_tf(i) for i in imgs]), labels)
    loss_180 = per_sample_ctc(crnn_r1, torch.stack([eval_tf(i.rotate(180)) for i in imgs]), labels)
    for img, fname, l0, l180 in zip(imgs, chunk.crop_filename, loss_0, loss_180):
        if l180 < l0:
            img = img.rotate(180); flipped.append(fname)
        img.save(os.path.join(ORIENTED_DIR, fname))

print(f"Crops turned 180°: {len(flipped)} / {len(yandex_df)} ({100 * len(flipped) / len(yandex_df):.0f}%)")
if flipped:
    fig, axes = plt.subplots(2, min(4, len(flipped)), figsize=(14, 3), squeeze=False)
    for j, f in enumerate(flipped[:axes.shape[1]]):
        show(axes[0, j], os.path.join(YANDEX_CROPS, f), "before")
        show(axes[1, j], os.path.join(ORIENTED_DIR, f), "after")
    plt.tight_layout(); save_plot("yandex_orientation_fix.png"); plt.show()

### Round 2: pretraining on the corrected crops

A fresh CRNN is trained on the corrected crops with the same settings. This is the **pretrained model**
that all Kigali experiments start from. Its validation score is the pretraining result for the report.

In [ ]:
seed_everything()
yo_train_loader = make_loader([os.path.join(ORIENTED_DIR, f) for f in y_train.crop_filename], y_train.label,
                              train_tf, batch_size=32, shuffle=True)
yo_val_loader = make_loader([os.path.join(ORIENTED_DIR, f) for f in y_val.crop_filename], y_val.label, eval_tf)

yo_train_eval = eval_subset_loader([os.path.join(ORIENTED_DIR, f) for f in y_train.crop_filename], y_train.label)
crnn_pre, hist_r2 = train_ctc(wm.CRNN(), yo_train_loader, yo_val_loader, epochs=EPOCHS["yandex_pretrain"],
                              lr=1e-3, name="crnn_yandex_pretrained", select="exact %", train_eval_loader=yo_train_eval)
plot_history(hist_r2, "Yandex round 2", "train_yandex_round2.png", metric="exact %")

pre_metrics = reading_metrics(*predict(crnn_pre, yo_val_loader))
print("\nYandex validation (pretraining result):", fmt(pre_metrics))
if pre_metrics["exact %"] < 30 and not FAST_RUN:
    print("WARNING: pretraining is much weaker than expected (about 85% exact). "
          "Check the label rebuild in Section 3.3 before continuing.")

## 9.3 Restore the lost leading zeros in the Kigali labels

Section 4.2 showed that many labels lost their leading zeros. Training on a mix of `565846` and
`00565846` for the same kind of picture would confuse the model, so we restore them. Three rules keep
this safe:

1. **Only zeros are ever added.** No digit is changed. The code proves it with a check.
2. **Labels I typed with leading zeros are never touched.** They are human ground truth.
3. **The method is tested before it is used.** We take the labels I typed myself, remove their leading
   zeros, and see which method puts back exactly the zeros I typed:
   - **Method A:** let the pretrained model choose how many zeros fit the picture best (lowest CTC loss),
     with at most 8 digits in total.
   - **Method B:** always pad to 8 digits.

Whichever method agrees with my own labels more often is used. Its agreement rate is reported, which
also shows how reliable leading zeros are in general.

In [ ]:
@torch.no_grad()
def pad_by_model(model, fname, digits, max_len=8):
    candidates = ["0" * k + digits for k in range(max(0, max_len - len(digits)) + 1)]
    img = eval_tf(Image.open(os.path.join(CROPS_DIR, fname)).convert("RGB")).unsqueeze(0)
    losses = per_sample_ctc(model, img.repeat(len(candidates), 1, 1, 1), candidates)
    return candidates[int(np.argmin(losses))]

crnn_pre.eval()
human = kigali_df[kigali_df.label_raw.str.startswith("0")].copy()
human["stripped"] = human.label_raw.str.lstrip("0")
human = human[human.stripped.str.len() > 0]
agree_a = np.mean([pad_by_model(crnn_pre, r.crop_filename, r.stripped, max(8, len(r.label_raw))) == r.label_raw
                   for r in human.itertuples()])
agree_b = np.mean([r.stripped.zfill(8) == r.label_raw for r in human.itertuples()])
USE_MODEL = agree_a >= agree_b
print(f"Hand-typed labels with leading zeros: {len(human)}")
print(f"Method A (model chooses): {100 * agree_a:.1f}% reproduce my zeros exactly")
print(f"Method B (always 8):      {100 * agree_b:.1f}% reproduce my zeros exactly")
print(f"Using: {'Method A' if USE_MODEL else 'Method B'}")

def normalise(r):
    raw = r.label_raw
    if raw.startswith("0") or len(raw) >= 8:
        return raw
    return pad_by_model(crnn_pre, r.crop_filename, raw, 8) if USE_MODEL else raw.zfill(8)

kigali_df["label"] = [normalise(r) for r in kigali_df.itertuples()]
assert all(n.endswith(r) and set(n[:len(n) - len(r)]) <= {"0"} for r, n in zip(kigali_df.label_raw, kigali_df.label))
typed = kigali_df.label_raw.str.startswith("0")
assert (kigali_df.loc[typed, "label"] == kigali_df.loc[typed, "label_raw"]).all()
print("\nChecks passed: only leading zeros were added, and my typed labels are unchanged.")
print(f"Labels changed: {(kigali_df.label != kigali_df.label_raw).sum()} / {len(kigali_df)}")
print("Label length after:", kigali_df.label.str.len().value_counts().sort_index().to_dict())

label_map = dict(zip(kigali_df.crop_filename, kigali_df.label))
for frame in (train_df, val_df, test_df):
    frame["label"] = frame.crop_filename.map(label_map)
kigali_df.to_csv(os.path.join(RESULTS_DIR, "kigali_labels_normalised.csv"), index=False)

## 9.4 Domain gap before fine-tuning

How well does the Yandex-trained model read **Kigali** meters it has never seen? This is the "before"
measurement of the domain gap. Expect a very low score: the meters, fonts, lighting and photo style are
different.

In [ ]:
k_val_loader = make_loader(kigali_paths(val_df), val_df.label, eval_tf)
zero_shot_val = reading_metrics(*predict(crnn_pre, k_val_loader), val_df.meter_group)
print("Yandex-pretrained CRNN on Kigali validation (no Kigali training):", fmt(zero_shot_val))

## 9.5 Learning Kigali meters: three experiments

All three use the same Kigali training set and are judged on the same validation set, so they can be
compared directly. They answer one question each:

| Experiment | Starts from | Trains on | Question it answers |
|---|---|---|---|
| **E1: from scratch** | Random weights | Kigali only | Can 310 Kigali photos alone teach the model? |
| **E2: fine-tune** | Yandex-pretrained | Kigali only | Does pretraining help? (E2 vs E1) |
| **E3: mixed fine-tune** | Yandex-pretrained | Kigali **+ Yandex** | Does mixing in Yandex reduce overfitting? (E3 vs E2) |

**Why mix in Yandex (E3)?** With only ~310 Kigali photos the model quickly starts memorising them:
training loss keeps falling while validation stops improving. Keeping some Yandex images in every batch
("rehearsal") forces the model to keep its general digit-reading skill while it adapts to Kigali meters.
Each epoch is about two-thirds Kigali and one-third Yandex.

The best epoch is chosen by **validation reading accuracy**. E1 has no early stopping: a model starting
from random weights can sit on the CTC "blank" plateau for many epochs before it starts reading, and
stopping it early would make the comparison with pretraining unfair.

In [ ]:
k_train_paths, k_train_labels = kigali_paths(train_df), train_df.label.tolist()
yandex_all_paths = [os.path.join(ORIENTED_DIR, f) for f in yandex_df.crop_filename]

def kigali_only_loader():
    return make_loader(k_train_paths, k_train_labels, train_tf, batch_size=16, shuffle=True)

def mixed_loader(kigali_share=0.65):
    kig = MeterDataset(k_train_paths, k_train_labels, train_tf)
    yan = MeterDataset(yandex_all_paths, yandex_df.label, train_tf)
    weights = [kigali_share / len(kig)] * len(kig) + [(1 - kigali_share) / len(yan)] * len(yan)
    sampler = WeightedRandomSampler(weights, num_samples=2 * len(kig), replacement=True,
                                    generator=torch.Generator().manual_seed(SEED))
    return DataLoader(ConcatDataset([kig, yan]), batch_size=16, sampler=sampler, num_workers=NUM_WORKERS,
                      pin_memory=PIN, collate_fn=ctc_collate)

def load_pretrained_crnn():
    m = wm.CRNN()
    m.load_state_dict(torch.load(os.path.join(MODELS_DIR, "crnn_yandex_pretrained.pt"), map_location="cpu"))
    return m

# The training curve of every Kigali experiment is measured on the same fixed slice of Kigali training
# photos (no augmentation), so the train/validation gap is comparable across experiments.
k_train_eval = eval_subset_loader(k_train_paths, k_train_labels)
FT = dict(val_groups=val_df.meter_group.values, train_eval_loader=k_train_eval)
experiments = {}

seed_everything()
experiments["E1 scratch"] = train_ctc(wm.CRNN(), kigali_only_loader(), k_val_loader,
                                      epochs=EPOCHS["kigali_scratch"], lr=1e-3, patience=None,
                                      name="crnn_kigali_scratch", **FT)
seed_everything()
experiments["E2 fine-tune"] = train_ctc(load_pretrained_crnn(), kigali_only_loader(), k_val_loader,
                                        epochs=EPOCHS["kigali_finetune"], lr=2e-4, backbone_prefix="cnn",
                                        backbone_lr=5e-5, freeze_epochs=5, patience=12,
                                        name="crnn_kigali_finetuned", **FT)
seed_everything()
experiments["E3 mixed"] = train_ctc(load_pretrained_crnn(), mixed_loader(), k_val_loader,
                                    epochs=EPOCHS["kigali_finetune"], lr=2e-4, backbone_prefix="cnn",
                                    backbone_lr=5e-5, freeze_epochs=5, patience=15,
                                    name="crnn_kigali_mixed", **FT)

In [ ]:
rows = [{"model": "EasyOCR (baseline)", **reading_metrics(easyocr_val, val_df.label, val_df.meter_group)}] \
    if easyocr_val is not None else []
rows.append({"model": "Yandex-pretrained, no Kigali training", **zero_shot_val})
for name, (model, hist) in experiments.items():
    rows.append({"model": name, **reading_metrics(*predict(model, k_val_loader), val_df.meter_group)})
val_table = pd.DataFrame(rows).round(2)
print("KIGALI VALIDATION (unseen meters)")
print(val_table.to_string(index=False))
val_table.to_csv(os.path.join(RESULTS_DIR, "kigali_validation_experiments.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
for name, (_, hist) in experiments.items():
    ax.plot(hist.epoch, hist["reading %"], label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("validation reading accuracy %"); ax.legend()
ax.set_title("Kigali experiments: validation reading accuracy")
plt.tight_layout(); save_plot("train_kigali_experiments.png"); plt.show()

**How to read this table**

- **Baselines vs everything else:** EasyOCR and the Yandex-only model show the size of the problem.
- **E2 vs E1:** the effect of pretraining. If E2 is clearly higher, pretraining on another country's
  meters is worth it even though those meters look different.
- **E3 vs E2:** the effect of rehearsal. Look at reading accuracy *and* length errors; mixing tends to
  reduce dropped or extra digits.
- The validation set has only about 34 meters, so a difference of a few points (2–3 photos) is noise.
  Look for differences that appear across *several* columns at once.

---
# 10. Error analysis

Accuracy alone does not say *how* the model fails. Here we sort every validation mistake of the mixed
model (E3) into types. This matters for billing: a wrong last digit is a few litres (almost nothing on a
bill), while a dropped digit can make a reading ten times too small.

In [ ]:
def error_type(pred, true):
    p, t = strip0(pred), strip0(true)
    if p == t:
        return "correct"
    if len(p) != len(t):
        return "dropped or extra digit"
    diffs = [i for i in range(len(t)) if p[i] != t[i]]
    if diffs == [len(t) - 1]:
        return "last digit only"
    return "one other digit" if len(diffs) == 1 else "two or more digits"

e3_model = experiments["E3 mixed"][0]
preds_e3, gts_e3 = predict(e3_model, k_val_loader)
errors = pd.DataFrame({"pred": preds_e3, "label": gts_e3})
errors["type"] = [error_type(p, g) for p, g in zip(preds_e3, gts_e3)]
print(errors["type"].value_counts().to_string())

confusions = Counter()
for p, g in zip(preds_e3, gts_e3):
    p, g = strip0(p), strip0(g)
    if len(p) == len(g):
        confusions.update((a, b) for a, b in zip(g, p) if a != b)
print("\nMost common digit confusions (true -> predicted):", confusions.most_common(8))
print("\nExamples of mistakes:")
print(errors[errors.type != "correct"].head(15).to_string(index=False))
errors.to_csv(os.path.join(RESULTS_DIR, "validation_errors_e3.csv"), index=False)

---
# 11. Alternative model and model selection

## 11.1 ResNet18 + BiGRU

For a fair comparison the alternative model goes through **the same two stages** as the CRNN: Yandex
pretraining (on the corrected crops), then mixed Kigali + Yandex fine-tuning with the same recipe as E3.
Its ResNet part starts from ImageNet weights. If those cannot be downloaded (Internet off), it starts
from random weights and says so.

In [ ]:
seed_everything()
try:
    resnet = wm.ResNetGRU(imagenet=True)
    print("ResNet18: ImageNet weights loaded")
except Exception as e:
    resnet = wm.ResNetGRU(imagenet=False)
    print("ResNet18: ImageNet weights unavailable, starting from random weights:", e)

resnet, hist_res_a = train_ctc(resnet, yo_train_loader, yo_val_loader, epochs=EPOCHS["resnet_pretrain"],
                               lr=3e-4, name="resnet_gru_yandex_pretrained", select="exact %",
                               train_eval_loader=yo_train_eval)
seed_everything()
resnet, hist_res_b = train_ctc(resnet, mixed_loader(), k_val_loader, epochs=EPOCHS["kigali_finetune"],
                               lr=2e-4, backbone_prefix="backbone", backbone_lr=5e-5, freeze_epochs=5,
                               patience=15, name="resnet_gru_kigali_mixed", **FT)
experiments["E4 ResNet18+BiGRU mixed"] = (resnet, hist_res_b)

## 11.3 Learning curves

A **learning curve** plots how a model improves epoch by epoch, on the training data and on the
validation data. Reading them:

- **Both losses fall and stay close:** the model is learning general features (good fit).
- **Training loss keeps falling while validation loss rises, and training accuracy pulls far above
  validation accuracy:** over-fitting; the model memorises the training meters. This is what
  rehearsal (E3, E4) and early stopping are meant to limit.
- **Both stay high or flat:** under-fitting, or the CTC "blank" plateau (expected for E1 in its first
  epochs, before it starts reading).

The dashed line is the epoch that was kept (best validation score). Training loss is measured with
augmentation during training; validation loss and both accuracy curves are measured without it, on
a fixed slice of Kigali training photos and on the validation set.


In [ ]:
curve_runs = {"Yandex pretraining (CRNN)": (hist_r2, "exact %"), **{n: (h, "reading %") for n, (_, h) in experiments.items()}}
fig, axes = plt.subplots(len(curve_runs), 2, figsize=(12, 3.4 * len(curve_runs)))
for row, (name, (h, metric)) in zip(axes, curve_runs.items()):
    plot_history(h, name, None, metric=metric, axes=row)
plt.tight_layout(); save_plot("learning_curves_readers.png"); plt.show()

# Over-fitting at the kept epoch: training minus validation accuracy (points).
gap = []
for name, (h, metric) in curve_runs.items():
    b = h.loc[h[metric].idxmax()]
    if f"train {metric}" in h:
        gap.append({"model": name, "metric": metric, "kept epoch": int(b.epoch),
                    "train %": round(b[f"train {metric}"], 1), "validation %": round(b[metric], 1),
                    "gap (points)": round(b[f"train {metric}"] - b[metric], 1),
                    "train loss": round(b["loss"], 3), "val loss": round(b["val loss"], 3)})
gap_table = pd.DataFrame(gap)
print(gap_table.to_string(index=False))
gap_table.to_csv(os.path.join(RESULTS_DIR, "learning_curve_gaps.csv"), index=False)


## 11.2 Choosing the model to deploy, on validation only

The rule is written down **before** looking at the test set:

1. Rank the candidates by **validation per-meter reading accuracy** (each meter counts once, so a few
   meters photographed many times cannot decide the result).
2. Any candidate within **3 points** of the best counts as equally good, because the validation set is
   small.
3. Among equally good candidates, pick the one with **fewer parameters** (cheaper and faster to run in
   the backend). If still tied, pick the higher reading accuracy.

The chosen model is recorded now. The test set in the next section then reports how good that choice
really is, but it cannot change the choice.

In [ ]:
ARCH = {"E1 scratch": "crnn", "E2 fine-tune": "crnn", "E3 mixed": "crnn", "E4 ResNet18+BiGRU mixed": "resnet_gru"}
FILES = {"E1 scratch": "crnn_kigali_scratch.pt", "E2 fine-tune": "crnn_kigali_finetuned.pt",
         "E3 mixed": "crnn_kigali_mixed.pt", "E4 ResNet18+BiGRU mixed": "resnet_gru_kigali_mixed.pt"}
cands = []
for name, (model, _) in experiments.items():
    m = reading_metrics(*predict(model, k_val_loader), val_df.meter_group)
    cands.append({"model": name, "params": n_params(model), **m})
cands = pd.DataFrame(cands)
best = cands["per-meter %"].max()
tied = cands[cands["per-meter %"] >= best - 3].sort_values(["params", "reading %"], ascending=[True, False])
SELECTED = tied.iloc[0]["model"]
print(cands.round(2).to_string(index=False))
print(f"\nSelected for deployment: {SELECTED}  (decided on validation, before the test set)")

---
# 12. Final evaluation on the locked test set

This is the only time the Kigali test set is used. It contains about 79 photos of 34 meters that **no**
model and **no** decision has seen. Every model is evaluated once, including the baselines, so the
whole story is visible in one table:

- **EasyOCR → Yandex-only CRNN → fine-tuned CRNN** shows the baseline and the domain gap before and
  after fine-tuning.
- **E1, E2, E3** show what pretraining and rehearsal contribute.
- **E4** is the alternative architecture.
- The **95% confidence interval** shows how uncertain each number is with only 34 meters. Overlapping
  intervals mean the difference is not statistically clear.

**After this cell, do not change any model or setting and re-run.** If you change something, the test
is no longer a fair test.

In [ ]:
test_loader = make_loader(kigali_paths(test_df), test_df.label, eval_tf)
test_groups = test_df.meter_group.values
test_preds = {}
easyocr_test = easyocr_predict(kigali_paths(test_df))
if easyocr_test is not None:
    test_preds["EasyOCR (baseline)"] = easyocr_test
test_preds["Yandex-pretrained, no Kigali training"] = predict(crnn_pre, test_loader)[0]
for name, (model, _) in experiments.items():
    test_preds[name] = predict(model, test_loader)[0]

rows, ok_flags = [], {}
for name, preds in test_preds.items():
    m = reading_metrics(preds, test_df.label, test_groups)
    ok = np.array([strip0(p) == strip0(g) for p, g in zip(preds, test_df.label)])
    lo, hi = bootstrap_ci(ok, test_groups)
    ok_flags[name] = ok
    rows.append({"model": name + ("  <- selected" if name == SELECTED else ""), **m, "95% CI": f"{lo:.0f}-{hi:.0f}"})
final_table = pd.DataFrame(rows).round(2)
print("=" * 100)
print(f"FINAL TEST RESULTS: {len(test_df)} photos of {test_df.meter_group.nunique()} meters never seen before")
print("=" * 100)
print(final_table.to_string(index=False))

print("\nReading accuracy % by verification status:")
by_status = pd.DataFrame({n: pd.Series(ok, index=test_df.index).groupby(test_df.verification_status).mean() * 100
                          for n, ok in ok_flags.items()}).round(1)
by_status["photos"] = test_df.verification_status.value_counts()
print(by_status.to_string())

final_table.to_csv(os.path.join(RESULTS_DIR, "final_test_results.csv"), index=False)
pred_out = test_df[["crop_filename", "label", "verification_status", "meter_group"]].copy()
for n, p in test_preds.items():
    pred_out[n] = p
pred_out.to_csv(os.path.join(RESULTS_DIR, "final_test_predictions.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(final_table))
ax.bar(x - 0.2, final_table["reading %"], 0.4, label="reading accuracy", color="#1f4e79")
ax.bar(x + 0.2, final_table["within-1 %"], 0.4, label="correct or within one digit", color="#7a9cc6")
ax.set_xticks(x); ax.set_xticklabels([n.replace("  <- selected", " *") for n in final_table.model], rotation=15, ha="right")
ax.set_ylabel("%"); ax.set_title("Held-out Kigali test set (* = model selected on validation)"); ax.legend()
plt.tight_layout(); save_plot("final_test_results.png"); plt.show()

**How to report these results honestly**

- Lead with the **selected model's per-meter reading accuracy and its confidence interval**, e.g.
  *"On 34 meters never seen during development, the model read X% of meters correctly (95% CI: A–B%)."*
- If the test score is much higher or lower than validation, say so. Both sets have only about 34 meters,
  so differences of 10–20 points between them are expected; the true accuracy likely lies in between.
- Report **within-one-digit accuracy** next to it, together with the error analysis (Section 10): most
  near misses are on the last litre wheel and have almost no effect on a bill.
- The Yandex pretraining score is a pretraining result, not the system's accuracy.

---
# 13. Stage 1: the quality gate

The quality gate looks at a photo and estimates whether it is **too poor to read** (dirt, condensation,
glare, wheels mid-turn). It is a yes/no classifier, with **unreadable** as the class we want to catch.

## 13.1 Three candidates, chosen on validation

| Candidate | Idea |
|---|---|
| **MobileNetV2 v1** | A small, fast image classifier pre-trained on ImageNet, fine-tuned on readable crops and the 68 real unreadable training crops |
| **MobileNetV2 v2** | Same, plus *synthetic* unreadable images: readable crops with fog, condensation, dirt, glare, blur, occlusion or ghosted "mid-turn" digits painted on |
| **Reader confidence** | No extra model: if the reader is unsure about its weakest digit, flag the photo |

Readable photos sometimes get *mild* damage during training and stay labelled readable. Otherwise the
model would learn "any visible artefact means unreadable".

**Important:** validation and test use **only real photos**. Synthetic images are only for training.

The candidate with the best **validation AUC** is chosen, and its decision threshold is also set on
validation. AUC (area under the ROC curve) measures how well a score separates the two classes: 0.5 is a
coin flip, 1.0 is perfect.

MobileNetV2 is chosen because it is designed for phones and small servers (about 2.2 million
parameters), so it can later run directly on the household's phone to give instant feedback.

In [ ]:
QG_SIZE = (128, 384)   # keeps the long shape of the number window
qg_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
qg_eval_tf = transforms.Compose([transforms.Resize(QG_SIZE), transforms.ToTensor(), qg_norm])
qg_aug_tf = transforms.Compose([transforms.Resize(QG_SIZE), transforms.ColorJitter(0.2, 0.2),
                                transforms.RandomApply([transforms.RandomRotation(5)], p=0.5),
                                transforms.ToTensor(), qg_norm])

# ---------- Realistic damage for synthetic unreadable images ----------
def _blobs(h, w, n, rmin, rmax, rng):
    m = np.zeros((h, w), np.float32)
    for _ in range(int(n)):
        cv2.ellipse(m, (int(rng.integers(0, w)), int(rng.integers(0, h))),
                    (int(rng.integers(rmin, rmax + 1)), int(rng.integers(max(2, rmin // 2), rmax + 1))),
                    float(rng.uniform(0, 180)), 0, 360, 1.0, -1)
    return m

def c_fog(img, rng, strong=True):
    h, w = img.shape[:2]
    nz = cv2.GaussianBlur(rng.random((h, w)).astype(np.float32), (0, 0), max(2, w / 8))
    nz = (nz - nz.min()) / (np.ptp(nz) + 1e-6)
    a = (rng.uniform(0.6, 0.85) if strong else rng.uniform(0.1, 0.25)) * (0.6 + 0.4 * nz)
    out = img.astype(np.float32) * (1 - a[..., None]) + float(rng.integers(190, 240)) * a[..., None]
    if strong:
        out = cv2.GaussianBlur(out, (0, 0), 2)
    return out.clip(0, 255).astype(np.uint8)

def c_condensation(img, rng):
    h, w = img.shape[:2]
    base = cv2.addWeighted(img, 0.3, cv2.GaussianBlur(img, (0, 0), max(3, w / 60)), 0.7, 0).astype(np.float32)
    m = cv2.GaussianBlur(_blobs(h, w, rng.integers(40, 90), max(2, h // 20), max(4, h // 6), rng), (0, 0), 1.5)[..., None]
    out = base * (1 - 0.5 * m) + 220 * 0.5 * m
    return (out * 0.75 + 255 * 0.25 * rng.uniform(0.5, 1.0)).clip(0, 255).astype(np.uint8)

def c_dirt(img, rng):
    h, w = img.shape[:2]
    m = np.zeros((h, w), np.float32)
    while m.mean() < 0.3:
        m = np.maximum(m, _blobs(h, w, rng.integers(2, 5), max(4, h // 5), max(8, h // 2), rng))
    m = (cv2.GaussianBlur(m, (0, 0), 3) * rng.uniform(0.8, 1.0)).clip(0, 1)[..., None]
    col = np.array([rng.integers(70, 120), rng.integers(50, 85), rng.integers(25, 55)], np.float32)
    return (img.astype(np.float32) * (1 - m) + col * m).clip(0, 255).astype(np.uint8)

def c_glare(img, rng):
    h, w = img.shape[:2]
    m = np.zeros((h, w), np.float32)
    cv2.ellipse(m, (int(rng.integers(w // 5, max(w // 5 + 1, 4 * w // 5))), int(rng.integers(h // 4, max(h // 4 + 1, 3 * h // 4)))),
                (int(rng.integers(max(2, w // 6), max(3, w // 3))), int(rng.integers(max(2, h // 3), max(3, h)))),
                float(rng.uniform(-20, 20)), 0, 360, 1.0, -1)
    m = cv2.GaussianBlur(m, (0, 0), max(3, w / 40))[..., None]
    return (img.astype(np.float32) * (1 - m) + 255 * m).clip(0, 255).astype(np.uint8)

def c_motion(img, rng):
    w = img.shape[1]
    k = int(rng.integers(max(9, w // 15), max(12, w // 6)))
    ker = np.zeros((k, k), np.float32); ker[k // 2, :] = 1
    ker = cv2.warpAffine(ker, cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(-30, 30)), 1), (k, k))
    return cv2.filter2D(img, -1, ker / max(ker.sum(), 1e-6))

def c_occlusion(img, rng):
    h, w = img.shape[:2]; out = img.copy()
    x0 = int(rng.integers(0, max(1, w // 2))); ww = int(rng.uniform(0.4, 0.7) * w)
    out[:, x0:min(w, x0 + ww)] = rng.integers(0, 60, 3) if rng.random() < 0.5 else rng.integers(120, 200, 3)
    return out

def c_midroll(img, rng):
    h, w = img.shape[:2]; out = img.astype(np.float32)
    shifted = np.roll(img, int(h * rng.uniform(0.35, 0.65)), axis=0).astype(np.float32)
    x0 = int(rng.integers(0, max(1, w // 3))); x1 = min(w, x0 + int(rng.uniform(0.5, 0.9) * w))
    out[:, x0:x1] = 0.5 * out[:, x0:x1] + 0.5 * shifted[:, x0:x1]
    return out.clip(0, 255).astype(np.uint8)

DAMAGE = {"fog": c_fog, "condensation": c_condensation, "dirt": c_dirt, "glare": c_glare,
          "motion blur": c_motion, "occlusion": c_occlusion, "mid-turn": c_midroll}

def strong_damage(img, rng):
    for n in rng.choice(list(DAMAGE), size=2 if rng.random() < 0.3 else 1, replace=False):
        img = DAMAGE[n](img, rng)
    return img

def mild_damage(img, rng):
    r = rng.random()
    if r < 0.4:
        return c_fog(img, rng, strong=False)
    if r < 0.8:
        return cv2.GaussianBlur(img, (0, 0), rng.uniform(0.6, 1.2))
    k = int(rng.integers(3, 6)); ker = np.zeros((k, k), np.float32); ker[k // 2, :] = 1 / k
    return cv2.filter2D(img, -1, ker)

rng_demo = np.random.default_rng(SEED)
demo = train_df.crop_filename.sample(min(3, len(train_df)), random_state=SEED).tolist()
fig, axes = plt.subplots(len(demo), len(DAMAGE) + 1, figsize=(22, 1.8 * len(demo)), squeeze=False)
for r, f in enumerate(demo):
    base = np.array(Image.open(os.path.join(CROPS_DIR, f)).convert("RGB"))
    axes[r, 0].imshow(base); axes[r, 0].set_title("original" if r == 0 else "")
    for c, (n, fn) in enumerate(DAMAGE.items(), 1):
        axes[r, c].imshow(fn(base.copy(), rng_demo)); axes[r, c].set_title(n if r == 0 else "")
for a in axes.ravel():
    a.axis("off")
plt.tight_layout(); save_plot("quality_gate_synthetic_damage.png"); plt.show()

In [ ]:
class QualityDataset(Dataset):
    # items: list of (crop_filename, label, mode); mode is 'plain', 'readable' (may get mild damage) or 'synthetic'.
    def __init__(self, items, tf, seed=None):
        self.items, self.tf = items, tf
        self.rng = np.random.default_rng(seed)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        f, y, mode = self.items[i]
        img = np.array(Image.open(os.path.join(CROPS_DIR, f)).convert("RGB"))
        rng = np.random.default_rng()
        if mode == "synthetic":
            img = strong_damage(img, rng)
        elif mode == "readable" and rng.random() < 0.4:
            img = mild_damage(img, rng)
        return self.tf(Image.fromarray(img)), y

def qg_items(frame):
    return [(f, int(y), "plain") for f, y in zip(frame.crop_filename, frame.y)]

qg_val_loader = DataLoader(QualityDataset(qg_items(qg_val), qg_eval_tf), batch_size=64, num_workers=NUM_WORKERS)
qg_test_loader = DataLoader(QualityDataset(qg_items(qg_test), qg_eval_tf), batch_size=64, num_workers=NUM_WORKERS)

@torch.no_grad()
def qg_scores(model, loader):
    model.eval()
    return torch.cat([torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu() for x, _ in loader]).numpy()

def new_mobilenet():
    try:
        m = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1)
    except Exception as e:
        print("MobileNetV2: ImageNet weights unavailable, random start:", e)
        m = torchvision.models.mobilenet_v2(weights=None)
    m.classifier[1] = nn.Linear(m.last_channel, 2)
    return m.to(DEVICE)

def train_quality_gate(items, name, epochs=None, freeze=3, patience=8):
    # Fixed, non-augmented training slice (real photos only) for the training AUC curve.
    global qg_train_eval_items, qg_train_eval_loader
    # Both classes, so AUC is defined: up to 200 readable and every distinct real unreadable photo.
    rng = np.random.default_rng(SEED)
    readable = sorted({f for f, y, mode in items if y == 0})
    unreadable = sorted({f for f, y, mode in items if y == 1 and mode != "synthetic"})
    readable = [readable[i] for i in rng.permutation(len(readable))[:200]]
    qg_train_eval_items = [(f, 0, "plain") for f in readable] + [(f, 1, "plain") for f in unreadable]
    qg_train_eval_loader = DataLoader(QualityDataset(qg_train_eval_items, qg_eval_tf), batch_size=64, num_workers=NUM_WORKERS)
    seed_everything()
    epochs = epochs or EPOCHS["quality_gate"]
    loader = DataLoader(QualityDataset(items, qg_aug_tf), batch_size=32, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=PIN)
    n_pos = sum(1 for it in items if it[1] == 1); n_neg = len(items) - n_pos
    weights = torch.tensor([len(items) / (2 * n_neg), len(items) / (2 * n_pos)], dtype=torch.float, device=DEVICE)
    model = new_mobilenet()
    crit = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW([{"params": model.features.parameters(), "lr": 5e-5},
                             {"params": model.classifier.parameters(), "lr": 3e-4}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    best, best_auc, bad, hist = None, -1, 0, []
    for ep in range(1, epochs + 1):
        for p in model.features.parameters():
            p.requires_grad = ep > freeze
        model.train(); losses = []
        for x, y in loader:
            loss = crit(model(x.to(DEVICE)), y.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        sched.step()
        auc = roc_auc_score(qg_val.y, qg_scores(model, qg_val_loader))
        # Learning-curve values: validation loss (same weighted loss) and AUC on a fixed training slice.
        model.eval()
        with torch.no_grad():
            vl = [crit(model(x.to(DEVICE)), y.to(DEVICE)).item() * len(y) for x, y in qg_val_loader]
        val_loss = sum(vl) / len(qg_val)
        train_auc = roc_auc_score([it[1] for it in qg_train_eval_items], qg_scores(model, qg_train_eval_loader))
        hist.append({"epoch": ep, "loss": np.mean(losses), "val loss": val_loss, "val AUC": auc, "train AUC": train_auc})
        if auc > best_auc:
            best_auc, best, bad = auc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        print(f"{name} | ep {ep:02d} | loss {np.mean(losses):.3f} | val AUC {auc:.3f}{'  *' if bad == 0 else ''}")
        if bad >= patience:
            print(f"{name} | early stop at epoch {ep}")
            break
    for p in model.parameters():
        p.requires_grad = True
    model.load_state_dict(best)
    torch.save(best, os.path.join(MODELS_DIR, f"{name}.pt"))
    return model, pd.DataFrame(hist)

items_v1 = [(f, 0, "plain") for f in train_df.crop_filename] + [(f, 1, "plain") for f in u_train.crop_filename]
items_v2 = ([(f, 0, "readable") for f in train_df.crop_filename] +
            [(f, 1, "plain") for f in u_train.crop_filename] * 3 +
            [(f, 1, "synthetic") for f in train_df.crop_filename])
print(f"v1 training items: {len(items_v1)} | v2 training items per epoch: {len(items_v2)}")

qg_v1, qg_hist_v1 = train_quality_gate(items_v1, "mobilenetv2_quality_gate_v1")
qg_v2, qg_hist_v2 = train_quality_gate(items_v2, "mobilenetv2_quality_gate_v2")

### Quality gate learning curves

Left: training and validation loss (the same class-weighted cross-entropy). Right: training and
validation AUC. A training AUC far above validation means the gate is memorising its 68 unreadable
training photos; a curve that is still rising when training stops would mean it needs more epochs.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for row, (name, h) in zip(axes, [("MobileNetV2 v1", qg_hist_v1), ("MobileNetV2 v2 (synthetic damage)", qg_hist_v2)]):
    best = h.loc[h["val AUC"].idxmax(), "epoch"]
    row[0].plot(h.epoch, h["loss"], label="training loss"); row[0].plot(h.epoch, h["val loss"], label="validation loss")
    row[0].set_title(f"{name}: loss"); row[0].set_xlabel("epoch")
    row[1].plot(h.epoch, h["train AUC"], label="training AUC"); row[1].plot(h.epoch, h["val AUC"], label="validation AUC")
    row[1].set_title(f"{name}: AUC"); row[1].set_xlabel("epoch"); row[1].set_ylim(0.4, 1.0)
    for a in row:
        a.axvline(best, ls="--", c="grey", lw=1, label=f"kept: epoch {best}"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); save_plot("learning_curves_quality_gate.png"); plt.show()


## 13.2 Reader confidence as a quality signal

The selected reader gives a probability for every digit it reads. When a wheel is dirty or mid-turn, that
probability should drop. We turn this into a score: **1 − (probability of the weakest digit)**, so a
higher score means "more likely unreadable", like the MobileNet scores.

In [ ]:
selected_model = experiments[SELECTED][0]

@torch.no_grad()
def reader_confidence(model, files):
    model.eval(); mins, preds = [], []
    for i in range(0, len(files), 64):
        x = torch.stack([eval_tf(Image.open(os.path.join(CROPS_DIR, f)).convert("RGB")) for f in files[i:i + 64]])
        for text, _, cmin in wm.decode_with_confidence(model(x.to(DEVICE))):
            mins.append(cmin); preds.append(text)
    return np.array(mins), preds

conf_val, _ = reader_confidence(selected_model, qg_val.crop_filename.tolist())
conf_test, _ = reader_confidence(selected_model, qg_test.crop_filename.tolist())

val_scores = {"MobileNetV2 v1": qg_scores(qg_v1, qg_val_loader),
              "MobileNetV2 v2 (+synthetic)": qg_scores(qg_v2, qg_val_loader),
              "Reader confidence": 1 - conf_val}
test_scores = {"MobileNetV2 v1": qg_scores(qg_v1, qg_test_loader),
               "MobileNetV2 v2 (+synthetic)": qg_scores(qg_v2, qg_test_loader),
               "Reader confidence": 1 - conf_test}

sel_rows, thresholds = [], {}
for name, s in val_scores.items():
    cand = np.unique(np.percentile(s, np.arange(1, 100)))
    f1s = [f1_score(qg_val.y, s >= t, zero_division=0) for t in cand]
    thresholds[name] = float(cand[int(np.argmax(f1s))])
    sel_rows.append({"method": name, "val AUC": roc_auc_score(qg_val.y, s), "val best F1 %": 100 * max(f1s)})
sel = pd.DataFrame(sel_rows).sort_values("val AUC", ascending=False)
QG_SELECTED = sel.iloc[0]["method"]
print(sel.round(3).to_string(index=False))
print(f"\nSelected on validation: {QG_SELECTED} (threshold {thresholds[QG_SELECTED]:.3f})")

## 13.3 Quality gate on the test set

Same rule as before: the choice is already made, the test only reports it. The table includes the blur
baseline. Two extra checks show what the gate means for the whole system:

- **Pipeline check:** if the gate works, the reader should be *more* accurate on the photos the gate
  accepts than on all photos.
- **Coverage vs accuracy:** if the system only reads automatically its most confident X% of photos and
  sends the rest to a person, how accurate is the automatic part? This is a practical way to present the
  system to WASAC.

In [ ]:
def qg_row(name, y, pred, score):
    return {"method": name, "accuracy %": 100 * accuracy_score(y, pred),
            "precision %": 100 * precision_score(y, pred, zero_division=0),
            "recall %": 100 * recall_score(y, pred, zero_division=0),
            "F1 %": 100 * f1_score(y, pred, zero_division=0), "AUC": roc_auc_score(y, score)}

lap_test = sharpness_of([os.path.join(CROPS_DIR, f) for f in qg_test.crop_filename])
rows = [qg_row("Blur threshold (baseline)", qg_test.y, (lap_test < LAP_THRESHOLD).astype(int), -lap_test)]
for name, s in test_scores.items():
    r = qg_row(name, qg_test.y, (s >= thresholds[name]).astype(int), s)
    r["method"] += "  <- selected" if name == QG_SELECTED else ""
    rows.append(r)
qg_table = pd.DataFrame(rows).round(3)
print(f"QUALITY GATE - TEST ({int((qg_test.y == 0).sum())} readable, {int((qg_test.y == 1).sum())} unreadable real photos)")
print(qg_table.to_string(index=False))
qg_table.to_csv(os.path.join(RESULTS_DIR, "quality_gate_test_results.csv"), index=False)

gate_pred = (test_scores[QG_SELECTED] >= thresholds[QG_SELECTED]).astype(int)
cm = confusion_matrix(qg_test.y, gate_pred, labels=[0, 1])
print(f"\nSelected gate: readable kept {cm[0,0]}, readable flagged {cm[0,1]}, "
      f"unreadable missed {cm[1,0]}, unreadable caught {cm[1,1]}")

n_read = len(test_df)
reader_ok = ok_flags[SELECTED]
accepted = gate_pred[:n_read] == 0
print("\nPipeline check (readable test photos):")
print(f"  reader accuracy on all photos:      {100 * reader_ok.mean():.1f}%")
if accepted.any():
    print(f"  reader accuracy on accepted photos: {100 * reader_ok[accepted].mean():.1f}% ({accepted.sum()} photos)")
if (~accepted).any():
    print(f"  reader accuracy on flagged photos:  {100 * reader_ok[~accepted].mean():.1f}% ({(~accepted).sum()} photos)")

order = np.argsort(conf_test[:n_read])[::-1]
cov = pd.DataFrame([{"automated %": int(c * 100), "photos": max(1, round(c * n_read)),
                     "reading accuracy %": 100 * reader_ok[order[:max(1, round(c * n_read))]].mean()}
                    for c in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]]).round(1)
print("\nAutomate only the most confident readings, send the rest to manual review:")
print(cov.to_string(index=False))
cov.to_csv(os.path.join(RESULTS_DIR, "coverage_vs_accuracy.csv"), index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["readable", "unreadable"]); ax[0].set_xlabel("predicted")
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["readable", "unreadable"]); ax[0].set_ylabel("true")
ax[0].set_title(f"Quality gate: {QG_SELECTED}")
grid = np.linspace(0.2, 1.0, 33)
ax[1].plot(grid * 100, [100 * reader_ok[order[:max(1, round(c * n_read))]].mean() for c in grid], color="#1f4e79")
ax[1].invert_xaxis(); ax[1].set_xlabel("% of readings automated (most confident first)")
ax[1].set_ylabel("reading accuracy %"); ax[1].set_title("Coverage vs accuracy (test)")
plt.tight_layout(); save_plot("quality_gate_results.png"); plt.show()

**How to interpret the quality gate**

With fewer than 100 real unreadable photos, expect all methods to be only moderately better than chance.
That is a real finding, not a failure:

- If the AUCs are all around 0.6–0.7, image-only quality checking is **data-limited**. In the app, the
  gate should then be a **soft prompt** ("this photo may be hard to read, retake?") that never blocks a
  submission.
- Misreads that slip through are caught later by **Stage 3** (anomaly detection on the household's
  history), which does not depend on photo quality at all.
- If synthetic damage (v2) scores lower than v1 on real photos, the model learned to spot the painted
  artefacts rather than real dirt and moisture. That is a known limitation of hand-made corruptions and
  worth reporting.

---
# 14. Export for deployment

Everything the FastAPI backend needs is collected in `/kaggle/working/deploy/`:

| File | Purpose |
|---|---|
| `meter_reader.pt` | Weights of the selected reading model |
| `quality_gate.pt` | Weights of the selected MobileNetV2 gate (if a MobileNet was selected) |
| `wasac_models.py` | The model code (CRNN, ResNetGRU, decoding, `MeterReader`) |
| `wasac_preprocessing.py` | The display crop, zero padding and m³ conversion |
| `model_config.json` | Architecture, input size, thresholds and the test results, so the backend never guesses |

The backend also needs the **Benax YOLOv8 `best.pt`** for cropping (it is an input to this notebook,
not produced by it).

The last check reloads the reader **from the exported files only**, exactly as the backend will, and
confirms it gives the same predictions as in the notebook.

In [ ]:
shutil.copy(os.path.join(MODELS_DIR, FILES[SELECTED]), os.path.join(DEPLOY_DIR, "meter_reader.pt"))
QG_FILES = {"MobileNetV2 v1": "mobilenetv2_quality_gate_v1.pt",
            "MobileNetV2 v2 (+synthetic)": "mobilenetv2_quality_gate_v2.pt"}
# If reader confidence was selected, still ship the better MobileNet (by validation AUC) for the camera prompt.
QG_MOBILENET = QG_SELECTED if QG_SELECTED in QG_FILES else \
    sel[sel.method.isin(list(QG_FILES))].iloc[0]["method"]
qg_file = QG_FILES[QG_MOBILENET]
shutil.copy(os.path.join(MODELS_DIR, qg_file), os.path.join(DEPLOY_DIR, "quality_gate.pt"))

sel_row = final_table[final_table.model.str.startswith(SELECTED)].iloc[0]
config = {
    "reader": {"arch": ARCH[SELECTED], "weights": "meter_reader.pt", "experiment": SELECTED,
               "input_size": [IMG_H, IMG_W], "grayscale": True, "normalize_mean": 0.5, "normalize_std": 0.5,
               "blank_index": wm.BLANK, "reading_length": 8, "decimals": 3},
    "quality_gate": {"arch": "mobilenet_v2", "weights": "quality_gate.pt", "input_size": list(QG_SIZE),
                     "normalize_mean": [0.485, 0.456, 0.406], "normalize_std": [0.229, 0.224, 0.225],
                     "positive_class": "unreadable", "selected_method": QG_SELECTED,
                     "shipped_model": QG_MOBILENET, "threshold": thresholds[QG_MOBILENET],
                     "use_as": "soft retake prompt, never blocking"},
    "cropper": {"yolo_weights": "best.pt (Benax YOLOv8, not included)", "conf": 0.15, "imgsz": 640, "padding": 0.05},
    "test_results": {"photos": int(len(test_df)), "meters": int(test_df.meter_group.nunique()),
                     "reading_accuracy_pct": float(sel_row["reading %"]), "per_meter_pct": float(sel_row["per-meter %"]),
                     "within_one_digit_pct": float(sel_row["within-1 %"]), "ci95": sel_row["95% CI"]},
    "created": time.strftime("%Y-%m-%d %H:%M"), "fast_run": FAST_RUN,
}
with open(os.path.join(DEPLOY_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

# Reload exactly as the backend will, and compare with the notebook's predictions
reader = wm.MeterReader(os.path.join(DEPLOY_DIR, "meter_reader.pt"), arch=ARCH[SELECTED], device=str(DEVICE))
check = test_df.head(5)
for f, nb_pred in zip(check.crop_filename, test_preds[SELECTED][:5]):
    out = reader.read(os.path.join(CROPS_DIR, f))
    assert out["raw_digits"] == nb_pred, "Exported model does not match the notebook"
    print(f"{f}: {out}")
shutil.rmtree(os.path.join(DEPLOY_DIR, "__pycache__"), ignore_errors=True)
print("\nExported reader matches the notebook.")
print("Deploy folder:", sorted(os.listdir(DEPLOY_DIR)))

---
# 15. Summary, limitations and next steps

The cell below prints the key numbers of this run in one place, ready to copy into the report.

In [ ]:
print("=" * 80)
print("RESULTS SUMMARY" + ("   (FAST_RUN - NOT FOR REPORTING)" if FAST_RUN else ""))
print("=" * 80)
print(f"Data: {len(kigali_df)} verified Kigali readings ({kigali_df.meter_group.nunique()} meters), "
      f"{len(unread_df)} unreadable crops, {len(yandex_df)} Yandex crops")
print(f"Split by meter: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test photos")
print(f"Yandex pretraining (validation): exact {pre_metrics['exact %']:.1f}% | edit distance {pre_metrics['edit dist']:.2f}")
print(f"Kigali label zeros: method {'A (model)' if USE_MODEL else 'B (pad to 8)'} reproduced "
      f"{100 * max(agree_a, agree_b):.1f}% of my hand-typed zeros")
print(f"\nSelected reader (chosen on validation): {SELECTED}")
print("\nFinal test (Kigali, unseen meters):")
print(final_table[["model", "reading %", "per-meter %", "95% CI", "within-1 %", "edit dist"]].to_string(index=False))
print(f"\nQuality gate selected on validation: {QG_SELECTED}")
print(qg_table[["method", "F1 %", "AUC"]].to_string(index=False))
print(f"\nAll outputs are in {WORK}: models/, plots/, results/, splits/, deploy/")

## Limitations

- **Small test set.** About 34 meters. The confidence intervals are wide, and validation and test scores
  can differ by 10–20 points. Results show feasibility, not production accuracy.
- **Label noise.** Leading zeros were partly lost before the data reached this notebook and were restored
  with a method that matches my own labels only about two times in three. This is why reading accuracy
  (ignoring leading zeros) is the headline metric.
- **One annotator.** All labels were verified by one person, so "unreadable" reflects one judgement.
- **Meter grouping by reading.** Photos are grouped by reading, not by a real meter ID. The rule is
  strict, but a meter ID from WASAC would make it exact.
- **Unreadable crops split at random.** They have no reading, so they cannot be grouped by meter.
- **Quality gate data.** Fewer than 100 real unreadable photos limits Stage 1 (Section 13).
- **Optimistic Yandex score.** The orientation fix used labels, including validation labels.

## Next steps

1. **WASAC data:** photos with a meter ID and date allow exact grouping, a larger test set, and real
   consumption histories for Stage 3.
2. **More unreadable examples** from field operations to strengthen the quality gate.
3. **Stage 3, anomaly detection,** is implemented in the backend (`backend/anomaly/`): it compares each
   new reading with the household's own history (consumption per day, ratio to the household's median,
   robust z-score), runs safety rules first, and uses an Isolation Forest once a household has at least
   three past readings. Its evaluation injects the error types measured in Section 10 into real reading
   histories once WASAC data is available.

## References

- Shi, B., Bai, X., & Yao, C. (2016). An end-to-end trainable neural network for image-based sequence
  recognition and its application to scene text recognition. *IEEE TPAMI*, 39(11), 2298–2304.
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L. C. (2018). MobileNetV2: Inverted residuals
  and linear bottlenecks. *CVPR*.
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.
- Graves, A., Fernández, S., Gomez, F., & Schmidhuber, J. (2006). Connectionist temporal classification.
  *ICML*.
- Liu, F. T., Ting, K. M., & Zhou, Z. H. (2012). Isolation-based anomaly detection. *ACM TKDD*, 6(1).
- Yandex Toloka Water Meters Dataset, CC BY-NC-ND 4.0 (Kaggle: tapakah68/yandextoloka-water-meters-dataset).

---
# 16. Compare this run and download the evidence

The app runs the models from the training run recorded in `backend/ml/model_config.json`. GPU training
varies slightly between runs, so this run may not give the same numbers. This section compares the two
and packs the charts and result tables into one zip file to download. The zip holds no model files.

**Keep the deployed models.** This run is for the learning curves. Do not replace the app's models
with the ones in `deploy/`, and keep reporting the test result of the deployed run.

In [ ]:
# The result of the training run the app is deployed with (backend/ml/model_config.json).
DEPLOYED_RUN = {"reading %": 55.7, "within-1 %": 70.9}

print(f"Selected in this run: {SELECTED}  (FAST_RUN = {FAST_RUN})")
print(f"{'metric':<14}{'deployed run':>14}{'this run':>12}{'difference':>12}")
for metric, deployed in DEPLOYED_RUN.items():
    now = float(sel_row[metric])
    print(f"{metric:<14}{deployed:>14.1f}{now:>12.1f}{now - deployed:>+12.1f}")
print("\nThe report keeps the deployed run's test result. This run supplies the learning curves.")

In [ ]:
EVIDENCE = "mubazi_notebook_evidence"
evidence_dir = os.path.join(WORK, EVIDENCE)
shutil.rmtree(evidence_dir, ignore_errors=True)
shutil.copytree(PLOTS_DIR, os.path.join(evidence_dir, "plots"))
shutil.copytree(RESULTS_DIR, os.path.join(evidence_dir, "results"))
archive = shutil.make_archive(evidence_dir, "zip", evidence_dir)
shutil.rmtree(evidence_dir)

print(f"Saved {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
print("Learning curves:", [f for f in sorted(os.listdir(PLOTS_DIR)) if f.startswith("learning_curves")])